<a href="https://colab.research.google.com/github/matias-ferrero/algo2-stats/blob/main/stats_algo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estadísticas — Algoritmos y Estructuras de Datos (FIUBA)

Notebook para generar estadísticas sobre las entregas de los alumnos de la cátedra Méndez/Pandolfo de Algoritmos y Estructuras de Datos (FIUBA), a partir de la planilla de corrección en Google Sheets.

**Cómo usarla:**

1. Ingresar a la notebook en Colab con la cuenta de Google que tenga acceso a la planilla de corrección.
2. Ir a **Planilla** y pegar la URL de la planilla del cuatrimestre a analizar.
3. Ejecutar todas las celdas de la notebook (`Entorno de ejecución > Ejecutar todas`). La primera ejecución puede tardar un poco más porque Colab tiene que conectar el entorno de ejecución.
4. Al ejecutar, el programa llega a un proceso de **Autenticación** (para más detalles, ver Configuración → Preprocesamiento). Va a aparecer un pop-up pidiendo iniciar sesión con una cuenta de Google que tenga acceso a la planilla, y otro para autorizar el acceso y la gestión de archivos de Drive (la planilla en Google Sheets). Es necesario que la cuenta tenga acceso de lectura a la planilla; si no, la notebook va a fallar más adelante.
5. Al terminar de ejecutarse, los gráficos y tablas de cada módulo aparecen en su subsección **Resultado**, dentro de **Estadísticas**.

**Cómo está organizada:**

- **Planilla**: la URL a analizar (lo único que se toca para correr sobre otro cuatrimestre).
- **Configuración**: todo el armado previo — imports, constantes (nombres de columna/hoja), el logger, y el preprocesamiento que se conecta a Google Sheets, baja cada hoja y la normaliza.
- **Estadísticas**: el análisis en sí — un módulo por cada pregunta que responde la notebook (por ejemplo, "¿cuántos aprobaron el TP0?"), agrupados por TP. Cada módulo tiene su **Implementación** (la función que arma el gráfico; en general no hace falta leerla) y su **Resultado** (donde se corre esa función y aparece el gráfico o la tabla).

Al abrir la notebook, todas las celdas están colapsadas, excepto la celda de la **Planilla** y las celdas de **Resultado** de cada módulo — la única excepción es la sección **ABB**, que arranca colapsada por completo (incluidos sus Resultado): esa hoja dejó de tomarse desde 2026, así que sus estadísticas quedan un clic más lejos por default. Se pueden expandir las celdas a gusto en caso de querer ver más detalles de la implementación, o sino colapsarlas en caso de querer ver solo ver celdas específicas. En general, no hace falta tocar nada de la notebook salvo completar la URL en la celda de **Planilla**.

## Planilla

Esta celda define **qué planilla de Google Sheets se analiza**.

- Tiene que ser la URL completa de una planilla de Google Sheets.
- Simplemente hay que copiar y pegar la URL de la planilla del cuatrimestre a analizar en el campo de abajo.
- La cuenta de Google con la que te autentiques en la sección **Autenticación** (para más información, ver Configuración → Preprocesamiento) necesita tener acceso de lectura a esa planilla; si no, la notebook va a fallar más adelante.
- Para analizar otro cuatrimestre alcanza con cambiar esta URL por la planilla correspondiente, y volver a ejecutar toda la notebook (`Entorno de ejecución > Ejecutar todas`).

In [ ]:
URL = 'https://docs.google.com/spreadsheets/d/1SAKzJt8NUDarH1r0asoiS7lWjLX-1TQ1LA28TWNrquk/edit?pli=1&gid=1495824376#gid=1495824376' #@param {type:"string"}

## Configuración

Todo el armado previo a calcular cualquier estadística: imports (**Dependencias**), nombres canónicos de columna/hoja (**Constantes**), el logger (**Logger**), y la conexión con Google Sheets más la normalización de los datos (**Preprocesamiento**). No hace falta tocar nada acá salvo que cambie el esquema de la planilla o se agregue una hoja nueva.

### Dependencias

Imports que usa el resto de la notebook: conectividad y autenticación con Google Sheets (`gspread`, `google.colab.auth`), el logger, `pandas` para manipular los datos, `matplotlib` para los gráficos, y `math` (trigonometría) para ubicar las etiquetas alrededor de los donuts.

In [ ]:
# Imports para conectividad y autenticación con Google Sheets
import gspread as gs # API de Python para manipular Google Sheets
from google.colab import auth # Maneja la autenticación del usuario en entornos de Colab
from google.auth import default # Obtiene las credenciales por defecto para acceder a la API
from google.auth.exceptions import GoogleAuthError # Maneja errores durante el proceso de autenticación

# Imports para el logger
import logging
import sys

# Imports para manipulación y análisis de datos
import pandas as pd
import unicodedata
import math # Trigonometría para ubicar etiquetas alrededor de los donuts (ver _draw_status_donut)

# Imports para visualización de datos y gráficos/figuras
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker # Formatea los ticks del eje Y como enteros prolijos

### Constantes

Nombres canónicos de columna (`COLUMNS`) y de hoja (`TP0_SHEET`, `TP1_SHEET`, `VALID_SHEETS`, `SHEET_ALIASES`, ...) que usa el resto de la notebook. Si la planilla real cambia un nombre de hoja o columna de un cuatrimestre a otro, es acá (o en `COLUMN_ALIASES`, en Normalización) donde hay que sumar el alias correspondiente.

In [ ]:
COLUMNS = ['Padron', 'intentos', 'Hora entrega', 'Estado', 'Corrector', 'Aprobado', 'Codigo', 'Pruebas', 'Informe', 'Interaccion', 'Nota final']

In [ ]:
TP0_SHEET = "TP0"
TP1_SHEET = "TP1"
LISTA_SHEET = "LISTA"
ABB_SHEET = "ABB"
HASH_SHEET = "HASH"
TP2_SHEET = "TP2"

# La planilla de 2026+ renombró la hoja HASH a DIC; ambas se refieren al mismo TP
SHEET_ALIASES = {"DIC": HASH_SHEET}

# Solo se procesan estas hojas; cualquier otra hoja (Karma, Parametros, Final, Asignaciones, ...) se ignora
VALID_SHEETS = {TP0_SHEET, TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET}

### Logger

Configura el logger que usa el resto de la notebook. Los mensajes están en español, y el nivel queda en `INFO`: en una corrida sin problemas no debería imprimirse ningún log, ya que los mensajes de camino feliz (autenticación exitosa, hoja convertida a DataFrame, columnas conservadas, ...) quedan en `debug`. Solo se ven `warning`/`error`/`critical`, y únicamente ante datos u hojas inesperados (una hoja que no existe, un valor de `Aprobado`/`Estado` que no matchea ninguna categoría conocida, etc.).

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
    force=True # Asegura que la configuración se aplique en el entorno de Colab
)
logger = logging.getLogger(__name__)

### Preprocesamiento

Se autentica contra Google Sheets, baja cada hoja de la planilla definida en **Planilla**, normaliza sus nombres de columna (ver **Normalización**, ya que cambian de un cuatrimestre a otro) y deja el resultado en `sheets`, listo para que **Estadísticas** lo use.

#### Autenticación

Establece una conexión entre `Google Colab` y la API de `Google Sheets`. Este proceso requiere ***verificación de identidad del usuario*** e inicializa un cliente para gestionar las planillas.

**Pasos para autenticarse:**

1. Iniciá el login ejecutando esta sección.
2. Va a aparecer una ventana emergente; permití el proceso de login.
3. Va a aparecer una ventana de inicio de sesión de Google; seleccioná una cuenta que tenga acceso a las planillas.
4. Otorgá los permisos para autorizar a Google Cloud SDK a ver y gestionar tus archivos de Google Drive y Google Sheets.
5. Verificación: si el login se completa sin errores no vas a ver ningún mensaje en la consola (esta celda solo loggea en `debug`, silencioso por defecto) y vas a poder correr el resto del script directamente. Si algo sale mal, ahí sí vas a ver un error explicando qué pasó.

In [ ]:
try:
    logger.debug("Iniciando proceso de autenticación...")

    # Abre el prompt de OAuth2 de Google para el usuario
    auth.authenticate_user()

    # Obtiene las credenciales del entorno y autoriza al cliente de gspread
    credential, _ = default()
    client = gs.authorize(credential)

    logger.debug("Autenticación exitosa: cliente inicializado.")

except GoogleAuthError as auth_err:
    # Maneja problemas específicamente relacionados al flujo de login o a los permisos
    logger.error(f"Error de autenticación: asegurate de completar el flujo de login.\nDetalles: {auth_err}")
    raise SystemExit("Ejecución detenida: acceso no autorizado.") from None

except Exception as e:
    # Problemas de red o fallas inesperadas del sistema
    logger.critical(f"Error inesperado durante la conexión: {e}")
    raise SystemExit("Ejecución detenida por un error crítico.") from None

#### Normalización

Distintos cuatrimestres nombraron las mismas columnas de manera diferente (por ejemplo, `Nota final` vs `Nota Final`, `intentos` vs `Intentos`, `Padron` vs `Columna 1`/` gv`, `status` vs `Estado`, o `Hora entrega` dividida en `Date` (UTC) + `Hora` (UTC-3)). Esta sección mapea cada variante conocida al nombre canónico usado en `COLUMNS` —usando siempre `Hora`, que está en horario local (UTC-3) igual que la `Hora entrega` de 2025, y no `Date`, que viene en UTC— para que el resto de la notebook no necesite preocuparse por qué versión de la planilla está leyendo.

In [ ]:
def _normalize_key(value):
    """Forma en minúsculas, sin acentos y sin espacios sobrantes, usada para matchear nombres de columna."""
    ascii_value = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')
    return ascii_value.strip().lower()

# Columnas que cambiaron de nombre entre cuatrimestres pero representan el mismo campo
COLUMN_ALIASES = {
    'gv': 'Padron',          # La hoja LISTA (2026+) usa " gv" en vez de "Padron"
    'columna 1': 'Padron',   # La hoja HASH (2025) usa "Columna 1" en vez de "Padron"
    'hora': 'Hora entrega',  # 2026+ dividió "Hora entrega" en "Date" (UTC) + "Hora" (UTC-3, la que se usaba antes)
    'status': 'Estado',      # Alias legible de la columna cruda "status" del pipeline de corrección
}

_CANONICAL_BY_KEY = {_normalize_key(col): col for col in COLUMNS}
_CANONICAL_BY_KEY.update(COLUMN_ALIASES)

def normalize_columns(df):
    """Renombra columnas para que tanto el formato de 2025 como el de 2026+ coincidan con COLUMNS."""
    rename = {
        col: _CANONICAL_BY_KEY[_normalize_key(col)]
        for col in df.columns
        if _normalize_key(col) in _CANONICAL_BY_KEY
    }
    return df.rename(columns=rename)

#### Hojas

Transforma todas las hojas disponibles en DataFrames para su posterior procesamiento y análisis.

- Ignora las hojas que no están en `VALID_SHEETS` (aplicando antes los alias de `SHEET_ALIASES`, por ejemplo `DIC` → `HASH`).
- Recorre cada hoja restante del archivo, usando la primera fila como encabezado y el resto como datos.
- Valida el contenido de cada hoja, distinguiendo entre datos activos, solo encabezados, o pestañas vacías.
- Detiene la ejecución si no se recolectaron datos (todas las hojas son inválidas).

In [ ]:
sh = client.open_by_url(URL)
csvs = sh.worksheets()

all_sheets = {}

for csv in csvs:
    title = SHEET_ALIASES.get(csv.title, csv.title)

    if title not in VALID_SHEETS:
        logger.debug(f"La hoja '{csv.title}' no es una de las hojas trackeadas. Se omite.")
        continue

    try:
        data = csv.get_all_values()

        if len(data) > 1:
            df = normalize_columns(pd.DataFrame(data[1:], columns=data[0]))
            all_sheets[title] = df

            logger.debug(f"La hoja '{csv.title}' se convirtió correctamente a DataFrame.")

        elif len(data) == 1:
            logger.warning(f"La hoja '{csv.title}' solo tiene encabezados y no tiene datos.")
        else:
            logger.debug(f"La hoja '{csv.title}' está completamente vacía.")

    except Exception as e:
        logger.error(f"No se pudo procesar la hoja '{csv.title}'.\nDetalles: {e}")

if not all_sheets:
    logger.error("No se pudo procesar ninguna hoja")
    raise SystemExit("Ejecución detenida: no se procesó ninguna hoja.")

#### Columnas

Filtra cada columna necesaria para el análisis, y solo conserva las hojas que contengan al menos una columna coincidente.

In [ ]:
sheets = {}

for title, df in all_sheets.items():
    columns = [col for col in COLUMNS if col in df.columns]

    if columns:
        sheets[title] = df[columns].copy()
        columns_str = ", ".join(columns)
        logger.debug(f"'{title}': columnas conservadas: [{columns_str}]")
    else:
        logger.warning(f"'{title}': no se encontró ninguna columna coincidente. Se omite.")

#### Vista previa

Imprime el comienzo de cada hoja después de filtrar las columnas.

In [ ]:
for title, df in sheets.items():
    print(f"Mostrando datos de la hoja: {title}")
    display(df.head(1))

## Estadísticas

A partir de `sheets` (ver Preprocesamiento), esta sección calcula y grafica las estadísticas de cada TP. **Utilidades** tiene los helpers compartidos por los gráficos; cada TP (`TP0`, `TP1`, `LISTA`, `ABB`, `HASH`, `TP2`, en el mismo orden cronológico en que se toman en la materia) agrupa sus propios módulos, y cada módulo tiene su **Implementación** (la función que arma el gráfico) y su **Resultado** (donde se corre esa función y se ve el gráfico o la tabla).

### Utilidades

Helpers compartidos por varios módulos de gráficos de esta sección, para no repetir la misma lógica en cada uno: formateo de etiquetas de torta/donut/barra de ratio (`fmt_labels`), dibujo de barras (`_draw_count_bar`), dibujo de donuts de estado (`_draw_status_donut` — incluye la lógica que saca del donut, con línea guía, la etiqueta de cualquier porción demasiado angosta para mostrarla adentro sin pisar a la vecina) y de barras de ratio (`_draw_ratio_bar`) junto con la paleta de estado fija que ambas usan (`STATUS_COLORS`: verde/ámbar/rojo + un cuarto tono gris para "N/A", ver más abajo), chequeo de qué hojas están disponibles (`hoja_disponible`), aviso de valores no reconocidos (`_warn_unmapped`), y el prefijo de numeración de figura de cada sección (`SECTION_FIGURE_PREFIX`).

Esta función actúa como un formateador personalizado para las etiquetas de datos de los gráficos de torta, donut y barra de ratio usados en esta sección. Toma el porcentaje relativo y el tamaño total de la muestra para calcular la cantidad exacta de estudiantes por porción o segmento de cada gráfico.

También suprime las etiquetas de 0% para mejorar la visualización de los gráficos, manteniéndolos legibles.

In [ ]:
def fmt_labels(pct, total):
    if pct == 0:
        return ""

    absolute = int(round(pct/100.*total))
    return f"{pct:.1f}%\n({absolute})"

Esta función dibuja, sobre un `ax` de matplotlib ya creado, una barra por cada categoría de `counts` con la misma rampa de color secuencial introducida en Notas del TP1 (más clara = menos alumnos, más oscura = más alumnos, según el ranking "dense" de la propia cantidad de alumnos de cada barra). La reutilizan tanto Notas del TP1 como los módulos que comparan varias distribuciones lado a lado en subplots (Notas por sección del TP1, Intentos en TP1), para no repetir esa lógica en cada uno. `students_count` normalmente es un escalar (el total sobre el que se calcula el `%` de cada barra, igual para todas), pero también acepta una `pd.Series` alineada al índice de `counts` cuando cada barra tiene su propio denominador — lo usa Abandonos entre TPs, donde cada barra es una transición distinta con su propia cantidad de aprobados del TP anterior.

In [ ]:
def _draw_count_bar(ax, counts, students_count, xlabel='Nota', title=None, colors=None):
    font_style = {'family': 'serif', 'style': 'italic'}
    ink_primary = '#0b0b0b'
    gridline_color = '#e1e0d9'
    baseline_color = '#c3c2b7'

    n = len(counts)

    if colors is not None:
        # Colores fijos por categoría (por ejemplo, la paleta de estado en Estado en TP1
        # Desaprobados): acá el color identifica una categoría, no una magnitud, así que
        # no tiene sentido la rampa secuencial de abajo.
        bar_colors = colors
    else:
        # Rampa secuencial (azul, claro -> oscuro), validada con --ordinal: se resuelve a
        # 5 escalones distinguibles. Cada barra se cuantiza según el ranking "dense" de su
        # propia cantidad de alumnos (no de la categoría): las categorías con menos
        # alumnos salen más claras, las que concentran más alumnos salen más oscuras. El
        # ranking dense hace que dos categorías con la misma cantidad de alumnos
        # compartan siempre el mismo color.
        shades = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281']
        dense_ranks = counts.rank(method='dense').astype(int) - 1
        n_distinct = dense_ranks.max() + 1 if n > 0 else 1
        bar_colors = [
            shades[min(len(shades) - 1, int(rank * len(shades) / n_distinct))]
            for rank in dense_ranks
        ]

    x_positions = range(n)
    bars = ax.bar(x_positions, counts.values, width=0.5, color=bar_colors, zorder=3)

    # students_count es normalmente un escalar (mismo total para todas las barras), pero
    # tambien puede ser una Series alineada a counts.index cuando cada barra tiene su
    # propio denominador (ver Abandonos entre TPs).
    totales = students_count if isinstance(students_count, pd.Series) else pd.Series(students_count, index=counts.index)

    for bar, count, total in zip(bars, counts.values, totales.reindex(counts.index).values):
        pct = (count / total * 100) if total > 0 else 0
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{pct:.0f}%\n({count})",
            ha='center', va='bottom', fontsize=9.5, weight='bold', color=ink_primary,
            family=font_style['family'], linespacing=1.3, zorder=4
        )

    # Los ticks de ambos ejes van en tinta primaria (no itálica) para que el contraste
    # contra el fondo sea alto: son datos que hay que leer, no texto de marca.
    ax.set_xticks(list(x_positions))
    ax.set_xticklabels(
        [f"{label:g}" if isinstance(label, (int, float)) else str(label) for label in counts.index],
        fontsize=11, color=ink_primary
    )

    max_count = counts.values.max() if n > 0 else 0
    # Con subtitulo (subplots comparando varias distribuciones) hace falta mas
    # aire arriba para que la etiqueta de la barra mas alta no choque con el titulo.
    headroom = 1.45 if title else 1.25
    ax.set_ylim(0, max_count * headroom + 1)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=6))
    for label in ax.get_yticklabels():
        label.set_fontsize(10.5)
        label.set_color(ink_primary)

    ax.grid(axis='y', color=gridline_color, linewidth=1, zorder=0)
    ax.set_axisbelow(True)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color(baseline_color)
        ax.spines[spine].set_linewidth(1)

    ax.set_xlabel(xlabel, fontproperties={**font_style, 'size': 12}, color=ink_primary)
    ax.set_ylabel('Cantidad de alumnos', fontproperties={**font_style, 'size': 12}, color=ink_primary)

    if title:
        ax.set_title(title, fontproperties={**font_style, 'size': 14, 'weight': 'bold'}, color=ink_primary, pad=16)

Paleta de estado fija, reutilizada por los gráficos de donut y de barra de ratio de esta sección. Los tres primeros colores (`good`/`warning`/`critical`) son los mismos que usa `tp1_estado_bar_chart` (ver Estado en TP1 Desaprobados) para OK/Timeout/Error: acá el color también identifica un estado (aprobó / desaprobó / sin resolver) y no una magnitud, así que tiene sentido reusar la misma paleta en vez de inventar una nueva. `neutral` (`#898781`, el mismo gris que ya usa esta notebook para "Muted (axis/labels)" en la paleta de referencia del skill `/dataviz`) lo usan los módulos "X Desaprobados" (y TP1 Aprobados) para **N/A** (el TP anterior no tiene ningún registro del alumno), distinto de **Sin Nota** (`warning`, ámbar: el alumno sí está pero esa corrección todavía está pendiente) — antes ambos casos compartían el mismo color porque compartían la misma categoría. Se eligió gris en vez del cuarto paso reservado (`serious`, `#ec835a`) porque ese naranja se lee casi igual al ámbar de `warning` a simple vista (el propio validador de `/dataviz` marca ese par por debajo del piso de visión normal, ΔE 13.6); el gris, al ser acromático, no se confunde con ningún otro color de la paleta y además comunica mejor "sin dato" que un estado más. Ver Caveats de datos en NOTAS_INTERNAS.md.

In [ ]:
STATUS_COLORS = {'good': '#0ca30c', 'warning': '#fab219', 'critical': '#d03b3b', 'neutral': '#898781'}

Esta función dibuja, sobre un `ax` de matplotlib ya creado, un gráfico de dona con una porción por categoría de `categories_count`, coloreada según `colors` (en el mismo orden que las categorías). A diferencia de la torta clásica que reemplaza, no usa `explode` ni sombra — separa las porciones con un borde fino en el color de la superficie (`edgecolor`), y aprovecha el hueco central para mostrar el total de alumnos en grande, en vez de dejarlo vacío. La reutilizan el módulo "Resultados" de cada hoja (TP1, LISTA, ABB, HASH, TP2) y TP1 Aprobados + todos los "X Desaprobados".

En vez de dejar que `autopct` ubique las etiquetas — que no evita que dos porciones vecinas angularmente angostas se superpongan entre sí (pasaba, por ejemplo, con Desaprobado y Sin Nota cuando ambas eran chicas) — cada porción se etiqueta a mano: si el arco de la porción mide al menos 25°, la etiqueta entra adentro como antes; si es más angosta, sale del donut con una línea guía fina, y las etiquetas que quedaron afuera se reparten verticalmente entre sí para no chocar (ver Convenciones de gráficos en NOTAS_INTERNAS.md). Con porciones parejas el resultado es idéntico al de antes; solo cambia cuando hay una o más porciones angostas.

In [ ]:
def _draw_status_donut(ax, categories, categories_count, students_count, colors):
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'
  surface = '#fcfcfb'

  patches, _texts = ax.pie(
      categories_count,
      labels=None,
      colors=colors,
      pctdistance=0.78,
      wedgeprops={'width': 0.55, 'edgecolor': surface, 'linewidth': 2}
  )

  # Etiqueta cada porción a mano (en vez de autopct) para poder detectar, una por una, cuáles
  # son angularmente muy angostas para que su etiqueta de 2 líneas entre sin pisar a la vecina.
  # Cada Wedge ya sabe su propio ángulo real (theta1/theta2, en grados) tal como lo dibujó
  # ax.pie, así que no hace falta recalcularlo a mano ni asumir en qué ángulo/sentido arrancó.
  UMBRAL_GRADOS = 25  # por debajo de esto, la etiqueta sale del donut con una línea guía
  etiquetas_afuera = []

  for patch, count in zip(patches, categories_count):
      pct = (count / students_count * 100) if students_count > 0 else 0
      span = patch.theta2 - patch.theta1
      texto = fmt_labels(pct, students_count)

      if not texto:
          continue

      rad = math.radians((patch.theta1 + patch.theta2) / 2)
      x, y = math.cos(rad), math.sin(rad)

      if span >= UMBRAL_GRADOS:
          # Entra cómoda adentro de la porción, como antes
          ax.text(
              x * 0.78, y * 0.78, texto,
              ha='center', va='center', fontsize=12, weight='bold', color='black',
              family=font_style['family'], linespacing=1.2, zorder=4
          )
      else:
          # Muy angosta: se saca afuera con línea guía; la posición vertical se ajusta
          # después para que no choque con otra etiqueta afuera (ver más abajo)
          etiquetas_afuera.append({'x': x, 'y': y, 'texto': texto})

  # Reparte verticalmente las etiquetas que quedaron afuera para que no se superpongan entre
  # sí: se ordenan de arriba hacia abajo y se empuja hacia abajo cualquiera que haya quedado
  # más cerca de la anterior que el paso mínimo.
  if etiquetas_afuera:
      etiquetas_afuera.sort(key=lambda e: e['y'], reverse=True)
      paso_minimo = 0.22
      for i in range(1, len(etiquetas_afuera)):
          if etiquetas_afuera[i - 1]['y'] - etiquetas_afuera[i]['y'] < paso_minimo:
              etiquetas_afuera[i]['y'] = etiquetas_afuera[i - 1]['y'] - paso_minimo

      for e in etiquetas_afuera:
          lado = 1 if e['x'] >= 0 else -1
          ax.annotate(
              e['texto'],
              xy=(e['x'], e['y']),
              xytext=(1.35 * lado, e['y']),
              ha='left' if lado > 0 else 'right', va='center',
              fontsize=12, weight='bold', color='black', family=font_style['family'], linespacing=1.2, zorder=4,
              arrowprops={'arrowstyle': '-', 'color': ink_primary, 'linewidth': 1, 'shrinkA': 0, 'shrinkB': 2}
          )

      # Deja aire para las etiquetas de afuera; sin esto, ax.pie() ajusta los límites
      # apenas por encima del radio 1 y el texto queda recortado fuera del eje.
      ax.set_xlim(-1.6, 1.6)
      ax.set_ylim(-1.3, 1.3)

  # El hueco del donut muestra el total en vez de quedar vacío
  ax.text(
      0, 0.08, f"{students_count}",
      ha='center', va='center', fontproperties={**font_style, 'size': 26, 'weight': 'bold'},
      color=ink_primary
  )
  ax.text(
      0, -0.16, "alumnos",
      ha='center', va='center', fontproperties={**font_style, 'size': 12},
      color=ink_primary
  )

  return patches

Esta función dibuja, sobre un `ax` de matplotlib ya creado, una única barra horizontal dividida en un segmento por categoría de `categories_count` (siempre 2, ver arriba), coloreada según `colors` en el mismo orden que las categorías. Cada segmento separa del vecino con un borde fino en el color de la superficie, igual que `_draw_status_donut`, y toda la barra lleva además un contorno fino en tinta primaria alrededor de todo el perímetro, para que se distinga del fondo blanco de la página (sin ese contorno, un segmento verde o rojo que toca el borde de la figura se pierde contra el blanco). La etiqueta de cada segmento (`%` + cantidad) se dibuja adentro si el segmento tiene ancho suficiente para que el texto entre con margen; si no, sale afuera del extremo más cercano en vez de recortarse. La reutilizan RESULTADOS TP0 y Abandonos post TP0, los dos módulos de 2 categorías donde antes había una torta de 2 porciones — un caso que `/dataviz` marca como anti-patrón (ver NOTAS_INTERNAS.md), ya que una proporción binaria se lee mejor como una única barra que como un círculo de dos porciones. El título y la caja de Referencias ya alcanzan para leer la cantidad y el porcentaje de cada categoría, así que estas dos funciones no necesitan un dato destacado aparte.

In [ ]:
def _draw_ratio_bar(ax, categories, categories_count, students_count, colors):
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'
  surface = '#fcfcfb'

  total = max(students_count, 1)
  left = 0
  bars = []

  for cat, count, color in zip(categories, categories_count, colors):
      bar = ax.barh(0, count, left=left, height=0.55, color=color, edgecolor=surface, linewidth=2, zorder=3)
      bars.append(bar[0])

      pct = (count / students_count * 100) if students_count > 0 else 0
      if pct > 0:
          label = f"{pct:.1f}%\n({count})"
          center = left + count / 2
          if pct >= 12:
              ax.text(center, 0, label, ha='center', va='center', fontsize=11, weight='bold',
                       color='black', family=font_style['family'], linespacing=1.2, zorder=4)
          else:
              outside_right = (left + count) > total / 2
              outside_x = (left + count) + total * 0.02 if outside_right else left - total * 0.02
              ax.text(outside_x, 0, label, ha='left' if outside_right else 'right', va='center',
                       fontsize=11, weight='bold', color=ink_primary, family=font_style['family'],
                       linespacing=1.2, zorder=4)

      left += count

  # Contorno fino alrededor de toda la barra, para que se distinga del fondo blanco
  ax.add_patch(plt.Rectangle((0, -0.275), total, 0.55, fill=False, edgecolor=ink_primary, linewidth=1.2, zorder=5))

  ax.set_xlim(0, total)
  ax.set_ylim(-0.7, 0.7)
  ax.axis('off')

  return bars

Antes de procesar cualquier sección de estadísticas, esta celda chequea qué hojas trackeadas están realmente presentes en la planilla (por ejemplo, `ABB` no existe desde 2026). Cada sección de estadísticas consulta `hoja_disponible` y se omite sin error si su hoja no está, en vez de fallar con `KeyError`. Al agregar una sección nueva a la notebook, alcanza con sumar su hoja acá.

El único módulo que además *usa* `hoja_disponible` para decidir *contra qué otra hoja* compararse (no solo si correr o no) es HASH Desaprobados: elige `ABB` si `hoja_disponible[ABB_SHEET]` es cierto, o cae a `LISTA` si no — ver el módulo "Resultado en el TP anterior de los desaprobados de HASH", más abajo.

In [ ]:
HOJAS_REQUERIDAS = [TP0_SHEET, TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET]

hoja_disponible = {hoja: hoja in sheets for hoja in HOJAS_REQUERIDAS}

for hoja, disponible in hoja_disponible.items():
    if not disponible:
        logger.warning(f"La hoja '{hoja}' no está presente en esta planilla; se omiten sus estadísticas.")

Esta función loggea un warning cuando algún valor de una columna categórica (por ejemplo `Aprobado` o `Estado`) no matcheó ninguna categoría conocida, algo que puede pasar si el vocabulario cambia de un cuatrimestre a otro (como ya pasó con `status`/`Estado` entre 2025 y 2026+). La usan los módulos que mapean esas columnas a categorías fijas.

In [ ]:
def _warn_unmapped(mapped, original, label):
    """Loggea un warning si algún valor de `original` no matcheó ninguna categoría conocida (quedó NaN en `mapped`)."""
    sin_mapear = mapped.isna()
    if sin_mapear.any():
        valores = sorted(set(original[sin_mapear].astype(str)))
        logger.warning(f"{label}: {sin_mapear.sum()} valores no reconocidos: {valores}")

Cada sección mayor de estadísticas (TP0, TP1, ...) tiene un prefijo fijo de numeración de figura, para que "Figura X.Y" sea consistente en toda la notebook (X identifica la sección, Y el orden de la figura dentro de ella). Al agregar una sección nueva alcanza con sumarle una entrada acá, siguiendo el próximo número consecutivo.

In [ ]:
SECTION_FIGURE_PREFIX = {'General': 1, 'TP0': 2, 'TP1': 3, 'LISTA': 4, 'ABB': 5, 'HASH': 6, 'TP2': 7}

### General

Estadísticas que combinan todas las hojas de la cursada (TP0 a TP2) en un solo análisis, en vez de mirar cada TP por separado — para eso están las secciones de abajo. Como estos módulos corren antes que las secciones por TP, no dependen de ninguna variable calculada ahí: recalculan todo lo que necesitan a partir de `sheets` y `hoja_disponible`.

#### Tasa de aprobación y desaprobación por TP

Este módulo compara, para cada TP disponible en la planilla, la proporción de estudiantes que aprobó, desaprobó, o todavía no tiene nota, para ver de un vistazo en qué TP el filtro es más duro o dónde se concentran las correcciones pendientes.

- Clasifica cada hoja con la misma lógica que su propio módulo "Resultados", a partir de `Nota final`.
- `TP0` no participa: no tiene `Nota final` (solo `Aprobado`, Si/No, sin categoría "Sin Nota" posible), así que compararlo en el mismo eje que el resto mezclaría dos criterios distintos. Queda afuera de este módulo — no de la sección General: sigue cubierto por el resto de la notebook (`RESULTADOS TP0`).
- Solo incluye las hojas presentes en la planilla (`hoja_disponible`), en el mismo orden cronológico que el resto de la notebook (TP1 → TP2).

Como resultado, el módulo genera una columna 100%-apilada por TP — con eje X (nombre del TP y cantidad de alumnos) y eje Y (porcentaje, con grilla) — con la paleta de estado fija (verde/rojo/ámbar) que ya usa el resto de la notebook, más una caja de Referencias con el total de alumnos analizados y el significado de cada color.

##### Implementación

Esta función dibuja una columna vertical 100%-apilada por TP (una barra por hoja, con eje X y eje Y explícitos), coloreada con la misma paleta de estado fija (`STATUS_COLORS`) que el resto de la notebook. No reutiliza `_draw_status_donut`: un donut por TP se probó primero, pero con 5 o 6 donuts comprimidos lado a lado, `ax.pie` fuerza aspecto cuadrado en cada subplot y no deja margen vertical para separar las etiquetas de las porciones angostas (pasaba, por ejemplo, con LISTA/ABB en el esquema 2025, donde Desaprobado y Sin Nota son ambas minoritarias). Tampoco reutiliza `_draw_count_bar`: acá el color identifica una categoría fija (estado), no una magnitud relativa, y las barras van apiladas en vez de una por categoría.

- Cada columna reparte sus tres categorías como segmentos apilados de 0 a 100%, con un contorno fino en tinta primaria alrededor de cada segmento (separador en el color de superficie, igual que `_draw_ratio_bar`).
- El eje Y muestra grilla y porcentaje (0% a 100%); el eje X muestra el nombre de cada TP y, debajo, su cantidad de alumnos.
- La etiqueta de cada segmento (`%` + cantidad) va adentro si el segmento mide al menos 10% de alto; si es más angosto, sale arriba de la columna con una línea guía, y las etiquetas que comparten una misma columna se separan horizontalmente entre sí para no superponerse (por ejemplo, Desaprobado y Sin Nota cuando ambas son minoritarias).
- La caja de Referencias es única y compartida (a la derecha de todas las columnas), con el total de alumnos analizados arriba y el significado de cada color debajo.

In [ ]:
def general_aprobacion_bars_chart(categories, tps_data, fig_num):
  n = len(tps_data)
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'
  surface = '#fcfcfb'
  gridline_color = '#e1e0d9'
  baseline_color = '#c3c2b7'
  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning']]
  UMBRAL_PCT = 10  # por debajo de esto, la etiqueta sale de la columna con una línea guía
  PASO = 16  # separación vertical entre etiquetas afuera apiladas de una misma columna

  plt.figure(figsize=(2.3 * n + 2.2, 6.8))
  ax = plt.gca()

  x_positions = list(range(n))
  legend_handles = []
  max_afuera = 0

  for x, (tp_name, categories_count, students_count) in zip(x_positions, tps_data):
      bottom = 0
      etiquetas_afuera = []

      for cat, color in zip(categories, colors):
          count = categories_count[cat]
          pct = (count / students_count * 100) if students_count > 0 else 0
          bar = ax.bar(x, pct, bottom=bottom, width=0.5, color=color, edgecolor=surface, linewidth=2, zorder=3)
          if x == 0:
              legend_handles.append(bar[0])

          if pct > 0:
              texto = f"{pct:.1f}%\n({count})"
              if pct >= UMBRAL_PCT:
                  # Entra cómodo centrado adentro del segmento
                  ax.text(
                      x, bottom + pct / 2, texto, ha='center', va='center', fontsize=9.5,
                      weight='bold', color='black', family=font_style['family'], linespacing=1.2, zorder=4
                  )
              else:
                  # Muy angosto: se guarda para ubicarlo arriba de la columna más abajo
                  etiquetas_afuera.append({'anchor_y': bottom + pct, 'texto': texto.replace(chr(10), ' ')})
          bottom += pct

      # Las etiquetas angostas de esta columna se apilan arriba de la barra (nunca adentro
      # de otro segmento) y se separan horizontalmente entre sí para no chocar, aunque haya
      # más de una porción angosta (ej. Desaprobado y Sin Nota, ambas minoritarias).
      m = len(etiquetas_afuera)
      max_afuera = max(max_afuera, m)
      for i, item in enumerate(etiquetas_afuera):
          y_off = 100 + 8 + i * PASO
          x_off = x + (i - (m - 1) / 2) * 0.4
          ax.annotate(
              item['texto'], xy=(x, item['anchor_y']), xytext=(x_off, y_off),
              ha='center', va='bottom', fontsize=8.5, weight='bold', color=ink_primary,
              family=font_style['family'], zorder=4,
              arrowprops={'arrowstyle': '-', 'color': ink_primary, 'linewidth': 0.8, 'shrinkA': 0, 'shrinkB': 2}
          )

  # Eje X: nombre del TP (negrita) y, debajo, su cantidad de alumnos
  ax.set_xticks(x_positions)
  ax.set_xticklabels(
      [f"{tp_name}\n({students_count} alumnos)" for tp_name, _, students_count in tps_data],
      fontsize=11, family=font_style['family'], color=ink_primary, linespacing=2.2
  )
  for label in ax.get_xticklabels():
      label.set_fontweight('bold')
  ax.tick_params(axis='x', length=0, pad=10)

  # Eje Y: 0% a 100% con grilla; el margen extra arriba deja lugar a las etiquetas afuera
  headroom = 100 + max_afuera * PASO + 18
  ax.set_ylim(0, headroom)
  ax.set_xlim(-0.65, n - 0.35)
  ax.set_yticks([0, 20, 40, 60, 80, 100])
  ax.set_yticklabels(['0%', '20%', '40%', '60%', '80%', '100%'], fontsize=10.5, color=ink_primary)
  ax.grid(axis='y', color=gridline_color, linewidth=1, zorder=0)
  ax.set_axisbelow(True)

  for spine in ['top', 'right']:
      ax.spines[spine].set_visible(False)
  for spine in ['left', 'bottom']:
      ax.spines[spine].set_color(baseline_color)
      ax.spines[spine].set_linewidth(1)

  ax.set_ylabel('% de alumnos', fontproperties={**font_style, 'size': 12}, color=ink_primary)

  plt.suptitle(
      'Tasa de aprobación y desaprobación por TP',
      x=0.42, y=0.97,
      fontproperties={**font_style, 'size': 19, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Caja de Referencias única y compartida: el color de cada categoría es el mismo en
  # todas las columnas, así que alcanza con mostrarla una sola vez, a la derecha. El total
  # analizado (suma de alumnos de todos los TPs de este gráfico) se agrega como primera
  # línea, ya que cada columna por separado ya lo muestra en su propio eje X.
  total_general = sum(students_count for _, _, students_count in tps_data)
  legend_labels = [f"Total analizado: {total_general} alumnos"] + list(categories)
  dummy_handle = plt.Line2D([], [], linestyle='none')
  legend_handles_final = [dummy_handle] + legend_handles

  ax.legend(
      legend_handles_final,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.03, 0.5),
      labelspacing=1.3,
      handletextpad=1.1,
      handlelength=1.3,
      borderpad=1.3
  )

  plt.figtext(
      0.42, 0.05, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.42, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.84, bottom=0.26, left=0.08, right=0.68)
  plt.show()

Esta sección recorre las hojas disponibles en el orden cronológico de la materia (TP1 → TP2, sin TP0) y clasifica cada una según su propia `Nota final`, replicando la misma lógica que ya usa cada módulo "Resultados" individual (ver TP1/LISTA/ABB/HASH/TP2), pero sin depender de las variables que esos módulos calculan más abajo en la notebook.

In [ ]:
if any(hoja_disponible[s] for s in [TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET]):
    general_aprobacion_categories = ['Aprobado', 'Desaprobado', 'Sin Nota']

    # TP0 no participa de este módulo: no tiene "Nota final" (solo "Aprobado", sin
    # categoría "Sin Nota" posible), así que compararlo en el mismo eje mezclaría dos
    # criterios distintos. Su propio resultado ya se ve en RESULTADOS TP0.
    general_aprobacion_secuencia = [TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET]

    general_aprobacion_tps_data = []
    for tp in general_aprobacion_secuencia:
        if not hoja_disponible[tp]:
            continue

        # Mismo criterio que el módulo "Resultados" de cada hoja (ver por ejemplo
        # Resultados del TP1): nota numérica = Aprobado, string "Desaprobado" =
        # Desaprobado, vacío o "Reentrega" = corrección todavía pendiente.
        resultado = sheets[tp]['Nota final'].apply(
            lambda nota: 'Desaprobado' if nota == 'Desaprobado' else ('Sin Nota' if nota in ('', 'Reentrega') else 'Aprobado')
        )

        # Reindexa a las 3 categorías fijas para que todas las hojas compartan siempre el
        # mismo esquema de colores/categorías, incluso si alguna tiene 0 en una de ellas.
        categories_count = resultado.value_counts().reindex(general_aprobacion_categories, fill_value=0)
        students_count = categories_count.sum()
        general_aprobacion_tps_data.append((tp, categories_count, students_count))

##### Resultado

In [ ]:
if any(hoja_disponible[s] for s in [TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET]):
    general_aprobacion_bars_chart(general_aprobacion_categories, general_aprobacion_tps_data, fig_num=f"{SECTION_FIGURE_PREFIX['General']}.1")

#### Análisis de notas

Este módulo muestra la distribución de notas numéricas de **todos los TPs juntos** (no una comparación TP por TP, sino el conjunto de todas las correcciones aprobadas de la cursada), para tener una foto global de cómo se distribuyen las notas más allá de una hoja puntual.

- Junta la `Nota final` (numérica, solo aprobados) de TP1, LISTA, ABB, HASH y TP2. `TP0` no tiene `Nota final` (solo `Aprobado`, Si/No), así que no participa acá — ver en cambio Tasa de aprobación y desaprobación por TP.
- Igual que Notas del TP1, siempre muestra todas las notas posibles del esquema vigente (detectado sobre el pool combinado), aunque nadie haya sacado esa nota.

Como resultado, el módulo genera un gráfico de barras con el porcentaje y la cantidad de correcciones que sacó cada nota, más el total y el promedio en la caja de Referencias.

##### Implementación

Reutiliza `_draw_count_bar` (ver Utilidades) tal cual, el mismo helper que usa Notas del TP1 y el resto de los módulos "Notas de X" — acá simplemente se le pasa el conteo agregado de las 5 hojas en vez del de una sola.

In [ ]:
def general_notas_bar_chart(notas_count, students_count, mean_nota, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, notas_count, students_count, xlabel='Nota')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Análisis de notas',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  promedio_txt = f"{mean_nota:.1f}" if students_count > 0 else "N/A"
  legend_labels = [f"Total: {students_count} correcciones aprobadas", f"Promedio: {promedio_txt}"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.6)
  plt.show()

Esta sección junta la `Nota final` de los aprobados de TP1/LISTA/ABB/HASH/TP2 (solo las hojas presentes en la planilla) en una única serie, detecta sobre ese pool combinado qué esquema de notas aplica (mismo criterio de "nota máxima observada" que usa Notas del TP1, recalculado acá porque esta sección corre antes), y cuenta cuántas correcciones sacaron cada nota.

In [ ]:
general_notas_sheets = [TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET]  # TP0 no tiene "Nota final"

if any(hoja_disponible[s] for s in general_notas_sheets):
    general_notas_valores = []
    for s in general_notas_sheets:
        if not hoja_disponible[s]:
            continue

        # Mismo filtro que Notas del TP1: solo aprobados (excluye "Desaprobado",
        # "Reentrega" y correcciones abiertas)
        notas_sheet = sheets[s]['Nota final']
        notas_sheet = notas_sheet[~notas_sheet.isin(['', 'Desaprobado', 'Reentrega'])]

        # Mismo formato de coma decimal que el resto de las notas
        general_notas_valores.append(pd.to_numeric(notas_sheet.str.replace(',', '.', regex=False)))

    general_notas = pd.concat(general_notas_valores, ignore_index=True)

    # Mismo criterio de inferencia de esquema que Notas del TP1 (0-2 antes de 2026, 4-10
    # desde 2026), recalculado acá porque este módulo corre antes que TP1 en la notebook.
    notas_pre_2026 = [round(i * 0.25, 2) for i in range(9)]
    notas_2026_en_adelante = [float(i) for i in range(4, 11)]
    max_nota = general_notas.max()
    general_notas_posibles = notas_2026_en_adelante if pd.isna(max_nota) or max_nota > 2 else notas_pre_2026

    # Cuenta las ocurrencias por nota, completando con 0 las notas posibles que nadie sacó
    general_notas_count = general_notas.value_counts().reindex(general_notas_posibles, fill_value=0)

    # Calcula el total y el promedio de todas las correcciones aprobadas
    general_notas_students_count = general_notas_count.sum()
    general_notas_mean = general_notas.mean()

##### Resultado

In [ ]:
if any(hoja_disponible[s] for s in general_notas_sheets):
    general_notas_bar_chart(general_notas_count, general_notas_students_count, general_notas_mean, fig_num=f"{SECTION_FIGURE_PREFIX['General']}.2")

#### Análisis por sección

Este módulo desglosa el Análisis de notas en sus tres componentes — `Codigo`, `Pruebas` e `Informe` — pero pooleando **todos los TPs juntos**, para ver si en general el rendimiento es parejo entre secciones a lo largo de toda la cursada, o si alguna concentra sistemáticamente las notas más bajas.

- Parte de la misma población que Análisis de notas (aprobados de TP1/LISTA/ABB/HASH/TP2) y del mismo esquema de notas ya detectado ahí, ya que las tres secciones puntúan en la misma escala que `Nota final`.
- `Interaccion` (la defensa oral del TP2) se excluye a propósito: solo existe en esa hoja, así que agregarla sería idéntico a mirar el TP2 solo — no aporta una comparación transversal real.

Como resultado, el módulo genera una figura con tres gráficos de barras — uno por sección, con el mismo eje Y para poder comparar alturas entre ellos — más el promedio de cada sección en la caja de Referencias.

##### Implementación

Arma una figura de tres subplots (uno por sección) y delega el dibujo de cada barra en `_draw_count_bar` (ver Utilidades), el mismo helper que usa Análisis de notas y Notas por sección del TP1 — así evita repetir tres veces la lógica de la rampa de color, la grilla y los ejes. Es prácticamente idéntica a `tp1_notas_por_seccion_bar_chart`, solo cambia el título y qué datos recibe.

In [ ]:
def general_secciones_bar_chart(secciones_counts, secciones_means, students_count, fig_num):
  secciones_labels = {'Codigo': 'Código', 'Pruebas': 'Pruebas', 'Informe': 'Informe'}

  fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), sharey=True)

  for ax, col in zip(axes, secciones_counts.keys()):
      _draw_count_bar(ax, secciones_counts[col], students_count, xlabel='Nota', title=secciones_labels[col])

  for ax in axes[1:]:
      ax.set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise a los
  # demas paneles; se recalcula una sola vez con el maximo real de las 3 secciones.
  max_count = max(counts.values.max() for counts in secciones_counts.values())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Análisis por sección',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  legend_labels = [f"Total: {students_count} correcciones aprobadas"] + [
      f"Promedio {secciones_labels[col]}: {mean:.1f}" if students_count > 0 else f"Promedio {secciones_labels[col]}: N/A"
      for col, mean in secciones_means.items()
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.06, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.01,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.22, right=0.83, wspace=0.18)
  plt.show()

Esta sección recorre las mismas hojas y el mismo filtro de aprobados que Análisis de notas, convierte `Codigo`, `Pruebas` e `Informe` a numérico (mismo formato de coma decimal) y cuenta cuántas correcciones sacaron cada nota en cada sección, reutilizando el `general_notas_posibles` ya calculado ahí (mismas escalas: 0-2 antes de 2026, 4-10 desde 2026, ya que las tres secciones puntúan igual que `Nota final`).

In [ ]:
if any(hoja_disponible[s] for s in general_notas_sheets):
    general_secciones = ['Codigo', 'Pruebas', 'Informe']
    general_secciones_valores = {col: [] for col in general_secciones}

    for s in general_notas_sheets:
        if not hoja_disponible[s]:
            continue

        # Mismo filtro que Análisis de notas: solo aprobados
        seccion_df = sheets[s][['Nota final', 'Codigo', 'Pruebas', 'Informe']].copy()
        seccion_df = seccion_df[~seccion_df['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

        for col in general_secciones:
            # Mismo formato de coma decimal que el resto de las notas
            general_secciones_valores[col].append(pd.to_numeric(seccion_df[col].str.replace(',', '.', regex=False)))

    general_secciones_data = {col: pd.concat(vals, ignore_index=True) for col, vals in general_secciones_valores.items()}

    # Reutiliza el esquema de notas ya detectado en Análisis de notas (mismas escalas que
    # "Nota final", ya que las tres secciones puntúan igual)
    general_secciones_counts = {
        col: serie.value_counts().reindex(general_notas_posibles, fill_value=0)
        for col, serie in general_secciones_data.items()
    }
    general_secciones_means = {col: serie.mean() for col, serie in general_secciones_data.items()}

    # Misma población que Análisis de notas (mismo filtro, mismas hojas)
    general_secciones_students_count = general_notas_students_count

##### Resultado

In [ ]:
if any(hoja_disponible[s] for s in general_notas_sheets):
    general_secciones_bar_chart(general_secciones_counts, general_secciones_means, general_secciones_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['General']}.3")

#### Abandonos entre TPs

Este módulo generaliza Abandonos post TP0 a toda la secuencia de la cursada (TP0 → TP1 → LISTA → ABB → HASH → TP2): mira, para cada transición entre TPs consecutivos, cuántos estudiantes **aprobaron** el TP anterior pero nunca llegaron a entregar el siguiente — en vez de duplicar esa misma sección una vez por cada par de hojas, las junta todas en un único gráfico comparativo.

- Arma la secuencia de hojas *presentes en la planilla*, en su orden cronológico, y toma los pares consecutivos de esa secuencia ya filtrada. Si una hoja intermedia falta (por ejemplo `ABB` desde 2026), el par salta el hueco automáticamente (queda `LISTA→HASH` en vez de fallar o inventar una comparación) — mismo criterio que ya usa HASH Desaprobados para elegir contra qué hoja anterior compararse.
- El resultado (aprobado o no) del TP anterior de cada par se calcula con la misma lógica que su propio módulo "Resultados": `Aprobado` para TP0, `Nota final` para el resto.
- "No entregar el siguiente" se mide por ausencia total de `Padron` en esa hoja, sin mirar su resultado — el mismo criterio que ya usa Abandonos post TP0.

Como resultado, el módulo devuelve un DataFrame con el detalle de cada abandono (padrón, TP anterior y TP siguiente), más un gráfico de barras con la cantidad y el porcentaje de abandonos de cada transición (el porcentaje es sobre los aprobados de *esa* transición, no sobre el total general).

##### Implementación

Reutiliza `_draw_count_bar` (ver Utilidades) igual que Análisis de notas, pero acá cada barra (transición) tiene su propio denominador — la cantidad de aprobados del TP anterior de *esa* transición, no un total único — así que le pasa la extensión de `students_count` como `pd.Series` (una por transición) en vez de un escalar. El color sigue siendo la rampa secuencial por defecto (más oscuro = más abandonos), ya que acá el color identifica una magnitud, no una categoría.

In [ ]:
def general_abandonos_bar_chart(counts, totales, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, counts, totales, xlabel='Transición')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Abandonos entre TPs',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  legend_labels = [
      f"Total de abandonos: {int(counts.sum())}",
      f"Aprobados analizados: {int(totales.sum())}",
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.55)
  plt.show()

Esta sección arma la secuencia dinámica de hojas disponibles y sus pares consecutivos, y para cada par calcula los aprobados del TP anterior que no aparecen en absoluto en el TP siguiente.

In [ ]:
general_secuencia_hojas = [s for s in [TP0_SHEET, TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET] if hoja_disponible[s]]
general_transiciones = list(zip(general_secuencia_hojas, general_secuencia_hojas[1:]))

if general_transiciones:
    general_abandonos_frames = []
    general_abandonos_counts = {}
    general_abandonos_totales = {}

    for prev, nxt in general_transiciones:
        if prev == TP0_SHEET:
            # TP0 no tiene "Nota final": aprobado se lee directo de "Aprobado"
            aprobados_prev = set(sheets[prev][sheets[prev]['Aprobado'] == 'Si']['Padron'])
        else:
            # Mismo criterio que el módulo "Resultados" de cada hoja: nota numérica cargada
            aprobados_prev = set(sheets[prev][~sheets[prev]['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]['Padron'])

        # "No entregó el siguiente" = ausencia total de Padron en esa hoja, sin mirar su
        # resultado — mismo criterio que ya usa Abandonos post TP0.
        padrones_nxt = set(sheets[nxt]['Padron'])
        padrones_abandonan = aprobados_prev - padrones_nxt

        transicion = f"{prev}→{nxt}"
        general_abandonos_totales[transicion] = len(aprobados_prev)
        general_abandonos_counts[transicion] = len(padrones_abandonan)

        if padrones_abandonan:
            general_abandonos_frames.append(pd.DataFrame({
                'Padron': sorted(padrones_abandonan),
                'TP Anterior': prev,
                'TP Siguiente': nxt,
            }))

    general_abandonos_counts = pd.Series(general_abandonos_counts)
    general_abandonos_totales = pd.Series(general_abandonos_totales)
    general_abandonos_df = (
        pd.concat(general_abandonos_frames, ignore_index=True) if general_abandonos_frames
        else pd.DataFrame(columns=['Padron', 'TP Anterior', 'TP Siguiente'])
    )

##### Resultado

In [ ]:
if general_transiciones:
    if len(general_abandonos_df) > 0:
        display(general_abandonos_df)
        print(f"\n")
        general_abandonos_bar_chart(general_abandonos_counts, general_abandonos_totales, fig_num=f"{SECTION_FIGURE_PREFIX['General']}.4")
    else:
        print("No hubo abandonos entre TPs consecutivos: todos los que aprobaron un TP entregaron el siguiente.")

### TP0

Estadísticas basadas en la hoja `TP0`, el primer trabajo práctico de la materia.

#### RESULTADOS TP0

Este módulo calcula la proporción de estudiantes que aprobaron y desaprobaron el TP0, el primer trabajo práctico de la materia.

- Clasifica a cada estudiante presente en la hoja `TP0` según su columna `Aprobado`, en **Aprobado** o **Desaprobado**. Todo estudiante presente en `TP0` ya tiene una corrección cerrada, así que no hace falta una tercera categoría de pendientes.

Como resultado, el módulo genera una barra de ratio para visualizar la proporción de cada categoría, con la cantidad y el porcentaje de cada una en su propio segmento y en la caja de Referencias.

##### Implementación

Esta función gestiona la generación y visualización completa de la barra de ratio de este módulo. Delega el dibujo del segmento y el contorno en `_draw_ratio_bar` (ver Utilidades) con la paleta de estado fija (`STATUS_COLORS`: verde para Aprobado, rojo para Desaprobado), y arma el título y la caja de Referencias — mismo mecanismo que el resto de los gráficos de esta sección. No lleva un dato destacado aparte: el título ya dice qué se mide, y el segmento más la caja de Referencias ya muestran la cantidad y el porcentaje de cada categoría.

In [ ]:
def tp0_results_ratio_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 3.8))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  bars = _draw_ratio_bar(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      bars,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados de TP0',
      x=0.5, y=0.85,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.18, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.06,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.72, bottom=0.34, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes de TP0 según su resultado. A diferencia de los módulos anteriores, no hace falta ningún cruce entre hojas: `Aprobado` se lee directamente de la propia hoja `TP0`, y esa columna no cambió de nombre entre el esquema de planilla de 2025 y el de 2026+, por lo que esta implementación funciona sin cambios contra cualquiera de los dos formatos.

In [ ]:
if hoja_disponible[TP0_SHEET]:
    tp0_results_categories = ['Aprobado', 'Desaprobado']

    tp0_results = sheets[TP0_SHEET][['Padron', 'Aprobado']].copy()

    # Mapea la columna "Aprobado" a las categorías
    tp0_results['Resultado'] = tp0_results['Aprobado'].map({'Si': 'Aprobado', 'No': 'Desaprobado'})
    _warn_unmapped(tp0_results['Resultado'], tp0_results['Aprobado'], "TP0: columna 'Aprobado'")
    tp0_results.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan ambas categorías (incluso si el conteo es 0)
    tp0_results_categories_count = tp0_results['Resultado'].value_counts().reindex(tp0_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp0_results_students_count = tp0_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP0_SHEET]:
    tp0_results_ratio_chart(tp0_results_categories, tp0_results_categories_count, tp0_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP0']}.1")

#### Abandonos post TP0

Este módulo identifica a los estudiantes que entregaron el TP0 pero nunca llegaron a entregar el TP1, es decir, bajas que ocurrieron entre ambos trabajos prácticos. Cruzando esos padrones con su resultado en el TP0, podemos estimar en qué proporción esas bajas se dieron después de aprobar el TP0 (posible abandono de la materia con un TP ya aprobado) o después de desaprobarlo (posible abandono tras la primera dificultad).

- Calcula la diferencia entre los padrones presentes en `TP0` y los presentes en `TP1`, para aislar a los estudiantes ausentes en TP1.
- Cruza esos padrones con su resultado (`Aprobado`) en TP0, clasificándolos en **TP0 Aprobado** o **TP0 Desaprobado**. Como estos estudiantes se seleccionan a partir de la propia hoja `TP0`, su corrección siempre está cerrada y no hace falta una tercera categoría de pendientes.
- Si todos los estudiantes que entregaron TP0 también entregaron TP1, el módulo lo informa por consola y no genera ningún gráfico.

Como resultado, el módulo devuelve un DataFrame con los datos de esos estudiantes, más una barra de ratio para visualizar la proporción de cada resultado, con la cantidad y el porcentaje de cada uno en su propio segmento y en la caja de Referencias.

##### Implementación

Esta función gestiona la generación y visualización completa de la barra de ratio de este módulo. Delega el dibujo del segmento y el contorno en `_draw_ratio_bar` (ver Utilidades) con la paleta de estado fija (`STATUS_COLORS`: verde para TP0 Aprobado, rojo para TP0 Desaprobado) — los mismos dos colores que usa RESULTADOS TP0, ya que ambos comparten el mismo significado de categoría. Arma el título y la caja de Referencias, igual que el resto de los gráficos de esta sección; no hace falta un dato destacado aparte porque el segmento y la caja de Referencias ya dicen cuántos abandonos habían aprobado el TP0.

In [ ]:
def missing_tp1_ratio_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 3.8))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  bars = _draw_ratio_bar(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      bars,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Abandonos post TP0',
      x=0.5, y=0.85,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.18, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.06,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.72, bottom=0.34, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección calcula la diferencia entre los padrones de TP0 y TP1 para aislar a los estudiantes ausentes en TP1, y categoriza su resultado en TP0. Tanto `Padron` como `Aprobado` son columnas comunes a ambos esquemas de planilla (2025 y 2026+): `Padron` ya queda normalizada por `normalize_columns` (a partir de `gv`/`Columna 1` según corresponda), y `Aprobado` no cambió de nombre entre cuatrimestres, por lo que esta implementación funciona sin cambios contra cualquiera de los dos formatos.

In [ ]:
if hoja_disponible[TP0_SHEET] and hoja_disponible[TP1_SHEET]:
    missing_tp1_categories = ['TP0 Aprobado', 'TP0 Desaprobado']

    padrones_en_tp0 = set(sheets[TP0_SHEET]['Padron'])
    padrones_en_tp1 = set(sheets[TP1_SHEET]['Padron'])
    padrones_ausentes = padrones_en_tp0 - padrones_en_tp1

    # Estudiantes que entregaron TP0 pero no TP1
    missing_tp1 = sheets[TP0_SHEET][sheets[TP0_SHEET]['Padron'].isin(padrones_ausentes)][['Padron', 'Aprobado']].copy()

    # Mapea la columna "Aprobado" a las categorías
    missing_tp1['TP0_Final'] = missing_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'})
    _warn_unmapped(missing_tp1['TP0_Final'], missing_tp1['Aprobado'], "Abandonos post TP0: columna 'Aprobado'")
    missing_tp1.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan ambas categorías (incluso si el conteo es 0)
    missing_tp1_categories_count = missing_tp1['TP0_Final'].value_counts().reindex(missing_tp1_categories, fill_value=0)

    # Calcula el total de estudiantes
    missing_tp1_students_count = missing_tp1_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP0_SHEET] and hoja_disponible[TP1_SHEET]:
    if missing_tp1_students_count > 0:
        display(missing_tp1)
        print(f"\n")
        missing_tp1_ratio_chart(missing_tp1_categories, missing_tp1_categories_count, missing_tp1_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP0']}.2")
    else:
        print("Todos los estudiantes que entregaron TP0 también entregaron TP1.")

### TP1

Estadísticas basadas en la hoja `TP1`, el segundo trabajo práctico de la materia. Varios módulos cruzan el resultado de TP1 con el de TP0, ya que el TP0 es un antecedente opcional (por RPL, un estudiante puede no tener registro en TP0).

#### Resultados del TP1

Este módulo calcula la proporción de estudiantes que aprobaron, desaprobaron, o todavía no tienen nota en el TP1, replicando el mismo análisis de RESULTADOS TP0 pero para el segundo trabajo práctico.

- Clasifica a cada estudiante presente en la hoja `TP1` según su columna `Nota final`, en **Aprobado** (si tiene una nota numérica cargada), **Desaprobado** (si el valor es el string "Desaprobado"), o **Sin Nota** (si `Nota final` está vacío o marcado como "Reentrega" — corrección todavía abierta, o pendiente de una nueva entrega). A diferencia de TP0, acá sí pueden quedar correcciones abiertas, por eso hace falta esta tercera categoría.

Como resultado, el módulo genera un donut para visualizar la proporción de cada categoría, con el total de alumnos en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo. Delega el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la paleta de estado fija (`STATUS_COLORS`: verde para Aprobado, rojo para Desaprobado, ámbar para Sin Nota) — el mismo esquema que usan TP1 Aprobados y TP1 Desaprobados, ya que las tres comparten la misma estructura de tres categorías. Arma el título y la caja de Referencias; el propio donut ya muestra el total en su hueco central.

In [ ]:
def tp1_results_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados de TP1',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes de TP1 según su resultado en `Nota final`. Al igual que en RESULTADOS TP0, no hace falta ningún cruce entre hojas: `Nota final` (2025) y `Nota Final` (2026+) coinciden vía `_normalize_key` sin necesitar un alias explícito, por lo que esta implementación funciona sin cambios contra cualquiera de los dos formatos.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_results_categories = ['Aprobado', 'Desaprobado', 'Sin Nota']

    tp1_results = sheets[TP1_SHEET][['Padron', 'Nota final']].copy()

    # Clasifica según "Nota final": el string "Desaprobado", vacío o "Reentrega" (corrección todavía abierta o pendiente de una nueva entrega), o una nota numérica
    tp1_results['Resultado'] = tp1_results['Nota final'].apply(
        lambda nota: 'Desaprobado' if nota == 'Desaprobado' else ('Sin Nota' if nota in ('', 'Reentrega') else 'Aprobado')
    )
    tp1_results.drop(columns=['Nota final'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    tp1_results_categories_count = tp1_results['Resultado'].value_counts().reindex(tp1_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp1_results_students_count = tp1_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_results_donut_chart(tp1_results_categories, tp1_results_categories_count, tp1_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.1")

#### Notas del TP1

Este módulo muestra la distribución de las notas numéricas obtenidas por quienes aprobaron el TP1. A diferencia de las demás secciones, acá no alcanza con una torta de dos o tres categorías: la nota final puede tomar varios valores distintos (incrementos de 0.25 entre 0 y 2 antes de 2026, o enteros entre 4 y 10 desde 2026), así que se usa un gráfico de barras con la cantidad de alumnos por cada nota.

- Toma únicamente los registros de `TP1` con una nota numérica cargada, descartando los desaprobados (`Nota final == "Desaprobado"`) y las correcciones todavía abiertas o en reentrega (`Nota final` vacío o "Reentrega").
- Siempre muestra **todas** las notas posibles del esquema vigente, aunque nadie haya sacado esa nota (con 0 alumnos), para que el gráfico no cambie de forma según qué notas salieron ese cuatrimestre.
- Cada barra se colorea con un degradé secuencial según su propia cantidad de alumnos (más clara = menos alumnos, más oscura = más alumnos), para que la nota más frecuente salte a la vista.
- La caja de **Referencias** reutiliza el mismo mecanismo (`plt.legend`) y los mismos parámetros de estilo que el resto de los gráficos de la notebook, con el total y el promedio en vez de categorías.

Como resultado, el módulo genera un gráfico de barras con el porcentaje y la cantidad de alumnos que sacó cada nota, más el total y el promedio en la caja de referencias.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de barras de este módulo. La rampa de color ordinal y el resto de la estética de cada barra viven en `_draw_count_bar` (ver Utilidades), reutilizado también por Notas por sección del TP1 e Intentos en TP1; esta función arma la figura, el título, y la caja de referencias con el total y el promedio.

In [ ]:
def tp1_notas_bar_chart(notas_count, students_count, mean_nota, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, notas_count, students_count, xlabel='Nota')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas de TP1',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Caja de referencias: mismo mecanismo (plt.legend) y mismos parámetros de estilo que
  # usan RESULTADOS TP0 / Abandonos post TP0 / APROBADOS-DESAPROBADOS TP1, para que las
  # figuras de la notebook compartan una única estética de caja de referencias en vez de
  # que esta tenga la suya propia. No hay categorías que mostrar con su color, así que
  # los handles son invisibles (handlelength=0) y solo quedan las dos líneas de texto.
  promedio_txt = f"{mean_nota:.1f}" if students_count > 0 else "N/A"
  legend_labels = [f"Total: {students_count} aprobados", f"Promedio: {promedio_txt}"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.6)
  plt.show()

Esta sección filtra a los estudiantes que aprobaron el TP1, cuenta cuántos sacaron cada nota completando con 0 las notas del esquema vigente que nadie sacó, y calcula el promedio y la mediana. Como los dos esquemas de notas (0 a 2 antes de 2026, 4 a 10 desde 2026) no se solapan, alcanza con mirar la nota máxima obtenida ese cuatrimestre para saber cuál de los dos aplica.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_notas = sheets[TP1_SHEET][['Padron', 'Nota final']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas)
    tp1_notas = tp1_notas[~tp1_notas['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    # Google Sheets devuelve los decimales con coma (ej. "0,75") en planillas con configuración regional en español
    tp1_notas['Nota'] = pd.to_numeric(tp1_notas['Nota final'].str.replace(',', '.', regex=False))

    # Antes de 2026 las notas iban de 0 a 2 en intervalos de 0.25; desde 2026 van de 4 a 10 en intervalos de 1.
    # Como los rangos no se solapan, la nota máxima observada alcanza para inferir qué esquema aplica.
    notas_pre_2026 = [round(i * 0.25, 2) for i in range(9)]
    notas_2026_en_adelante = [float(i) for i in range(4, 11)]
    max_nota = tp1_notas['Nota'].max()
    notas_posibles = notas_2026_en_adelante if pd.isna(max_nota) or max_nota > 2 else notas_pre_2026

    # Cuenta las ocurrencias por nota, completando con 0 las notas posibles que nadie sacó
    tp1_notas_count = tp1_notas['Nota'].value_counts().reindex(notas_posibles, fill_value=0)

    # Calcula el total y el promedio de los aprobados
    tp1_notas_students_count = tp1_notas_count.sum()
    tp1_notas_mean = tp1_notas['Nota'].mean()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_notas_bar_chart(tp1_notas_count, tp1_notas_students_count, tp1_notas_mean, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.2")

#### Notas por sección del TP1

Este módulo desglosa la nota final de quienes aprobaron el TP1 en sus tres componentes — `Codigo`, `Pruebas` e `Informe` — para ver si el rendimiento es parejo entre secciones o si alguna concentra las notas más bajas.

- Parte de la misma población que Notas del TP1 (aprobados con nota numérica cargada) y del mismo esquema de notas ya detectado para `Nota final`, ya que las tres secciones puntúan en la misma escala (0 a 2 antes de 2026, 4 a 10 desde 2026).
- Al igual que Notas del TP1, siempre muestra todas las notas posibles del esquema vigente en cada sección, aunque nadie haya sacado esa nota, para que el gráfico no cambie de forma de un cuatrimestre a otro.

Como resultado, el módulo genera una figura con tres gráficos de barras — uno por sección, con el mismo eje Y para poder comparar alturas entre ellos — más el promedio de cada sección en la caja de referencias.

##### Implementación

Esta función arma una figura de tres subplots (uno por sección) y delega el dibujo de cada barra en `_draw_count_bar` (ver Utilidades), el mismo helper que usa Notas del TP1 — así evita repetir tres veces la lógica de la rampa de color, la grilla y los ejes. Solo agrega el título de figura, el eje Y compartido, y una caja de referencias única con el promedio de cada sección.

In [ ]:
def tp1_notas_por_seccion_bar_chart(secciones_counts, secciones_means, students_count, fig_num):
  secciones_labels = {'Codigo': 'Código', 'Pruebas': 'Pruebas', 'Informe': 'Informe'}

  fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), sharey=True)

  for ax, col in zip(axes, secciones_counts.keys()):
      _draw_count_bar(ax, secciones_counts[col], students_count, xlabel='Nota', title=secciones_labels[col])

  for ax in axes[1:]:
      ax.set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise a los
  # demas paneles; se recalcula una sola vez con el maximo real de las 3 secciones.
  max_count = max(counts.values.max() for counts in secciones_counts.values())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas por sección del TP1',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas del TP1), con un
  # promedio por sección en vez de uno solo.
  legend_labels = [f"Total: {students_count} aprobados"] + [
      f"Promedio {secciones_labels[col]}: {mean:.1f}" if students_count > 0 else f"Promedio {secciones_labels[col]}: N/A"
      for col, mean in secciones_means.items()
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.06, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.01,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.22, right=0.83, wspace=0.18)
  plt.show()

Esta sección filtra a los mismos aprobados que Notas del TP1, convierte `Codigo`, `Pruebas` e `Informe` a numérico (mismo formato de coma decimal), y cuenta cuántos alumnos sacaron cada nota en cada sección, reutilizando el `notas_posibles` ya calculado para `Nota final`.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_secciones = sheets[TP1_SHEET][['Padron', 'Nota final', 'Codigo', 'Pruebas', 'Informe']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas), mismo filtro que tp1_notas
    tp1_secciones = tp1_secciones[~tp1_secciones['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    secciones = ['Codigo', 'Pruebas', 'Informe']
    for col in secciones:
        # Mismo formato de coma decimal que el resto de las notas
        tp1_secciones[col] = pd.to_numeric(tp1_secciones[col].str.replace(',', '.', regex=False))

    # Reutiliza el esquema de notas ya detectado para "Nota final" (mismas escalas: 0-2 antes
    # de 2026, 4-10 desde 2026), ya que las tres secciones puntúan en la misma escala
    tp1_secciones_counts = {
        col: tp1_secciones[col].value_counts().reindex(notas_posibles, fill_value=0)
        for col in secciones
    }
    tp1_secciones_means = {col: tp1_secciones[col].mean() for col in secciones}
    tp1_secciones_students_count = len(tp1_secciones)

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_notas_por_seccion_bar_chart(tp1_secciones_counts, tp1_secciones_means, tp1_secciones_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.3")

#### Resultado en TP0 de los aprobados de TP1

Este módulo analiza el desempeño previo (opcional) en el TP0 de quienes aprobaron el TP1. Es el complemento de TP1 Desaprobados: en vez de mirar por qué fallan, mira si venir con el TP0 aprobado (o directamente sin TP0, vía RPL) se traduce en mejores resultados en el TP1.

- Aísla a la población de estudiantes que aprobó el TP1 y, a partir de esos registros, cruza sus acciones en el TP0 para determinar si aprobaron, desaprobaron, todavía no tienen nota, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en el TP1 pero ausentes en los registros del TP0, y los clasifica como *"No disponible"* (N/A). Esto es distinto de **Sin Nota**: un estudiante que sí está en TP0 pero cuyo `Aprobado` no matchea ninguna categoría (por ejemplo, esa corrección todavía no cerró) — antes ambos casos se mezclaban bajo N/A; separarlos evita confundir "no entregó TP0" con "entregó TP0 pero todavía no tiene resultado".
- También asegura que el análisis siempre considere los cuatro resultados posibles del TP0 (**Aprobado, Desaprobado, Sin Nota, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si la hoja `TP0` no está disponible en esta planilla (a diferencia del resto de las secciones, que se omiten enteras cuando falta la hoja que necesitan — ver `hoja_disponible` en Utilidades), este módulo igual corre: todos los estudiantes caen directamente en **TP0 N/A**, igual que si cada uno individualmente no tuviera registro en TP0.

Como resultado, el módulo devuelve un DataFrame con los datos importantes de los estudiantes, más un donut para una mejor visualización de los números, con el total de aprobados en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo, delegando el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la misma paleta de estado (`STATUS_COLORS`) que Resultados del TP1 y TP1 Desaprobados. Es prácticamente idéntica a la de TP1 Desaprobados (mismas 4 categorías y colores), solo cambia el título y el número de figura.

In [ ]:
def passed_tp1_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning'], STATUS_COLORS['neutral']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultado en TP0 de los aprobados de TP1',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que aprobaron y categoriza esos registros, espejando la lógica de TP1 Desaprobados.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    passed_tp1_categories = ['TP0 Aprobado', 'TP0 Desaprobado', 'TP0 Sin Nota', 'TP0 N/A']

    # Filtra a los estudiantes que aprobaron el TP1 (excluye "Desaprobado", "Reentrega" y correcciones abiertas)
    passed_tp1 = sheets[TP1_SHEET][~sheets[TP1_SHEET]['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])].copy()

    # Cruza con los datos de TP0 usando "Padron" y conservando "Aprobado". Si la hoja TP0 no
    # está disponible en esta planilla, se usa un DataFrame vacío: el merge no encuentra
    # coincidencias y todos los estudiantes caen en "TP0 N/A" más abajo.
    tp0_df = sheets.get(TP0_SHEET, pd.DataFrame(columns=['Padron', 'Aprobado']))[['Padron', 'Aprobado']]
    padrones_en_tp0 = set(tp0_df['Padron'])
    passed_tp1 = passed_tp1.merge(tp0_df, on='Padron', how='left')

    # Mapea la columna "Aprobado" a las categorías. Si el estudiante está en TP0 pero su
    # "Aprobado" no matchea ninguna categoría (corrección sin cerrar, valor no reconocido), es
    # "TP0 Sin Nota"; si directamente no aparece en TP0, es "TP0 N/A".
    passed_tp1['TP0_Final'] = passed_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'})
    en_tp0_sin_mapear = passed_tp1['TP0_Final'].isna() & passed_tp1['Padron'].isin(padrones_en_tp0)
    passed_tp1.loc[en_tp0_sin_mapear, 'TP0_Final'] = 'TP0 Sin Nota'
    passed_tp1['TP0_Final'] = passed_tp1['TP0_Final'].fillna('TP0 N/A')
    passed_tp1.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 4 categorías (incluso si el conteo es 0)
    passed_tp1_categories_count = passed_tp1['TP0_Final'].value_counts().reindex(passed_tp1_categories, fill_value=0)

    # Calcula el total de estudiantes
    passed_tp1_students_count = passed_tp1_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    passed_tp1_donut_chart(passed_tp1_categories, passed_tp1_categories_count, passed_tp1_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.4")

#### Resultado en TP0 de los desaprobados de TP1

Este módulo analiza el desempeño previo (opcional) en el TP0 de quienes desaprobaron el TP1. Esto puede darnos una idea de por qué tantos estudiantes fallan en esta etapa, dado que lo único que tienen antes es el TP0 (aparte de RPL).

- Aísla a la población de estudiantes que desaprobó el TP1 y, a partir de esos registros, cruza sus acciones en el TP0 para determinar si aprobaron, desaprobaron, todavía no tienen nota, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en el TP1 pero ausentes en los registros del TP0, y los clasifica como *"No disponible"* (N/A). Esto es distinto de **Sin Nota**: un estudiante que sí está en TP0 pero cuyo `Aprobado` no matchea ninguna categoría (por ejemplo, esa corrección todavía no cerró) — antes ambos casos se mezclaban bajo N/A; separarlos evita confundir "no entregó TP0" con "entregó TP0 pero todavía no tiene resultado".
- También asegura que el análisis siempre considere los cuatro resultados posibles del TP0 (**Aprobado, Desaprobado, Sin Nota, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si la hoja `TP0` no está disponible en esta planilla (a diferencia del resto de las secciones, que se omiten enteras cuando falta la hoja que necesitan — ver `hoja_disponible` en Utilidades), este módulo igual corre: todos los estudiantes caen directamente en **TP0 N/A**, igual que si cada uno individualmente no tuviera registro en TP0.

Como resultado, el módulo devuelve un DataFrame con los datos importantes de los estudiantes, más un donut para una mejor visualización de los números, con el total de desaprobados en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo, delegando el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la misma paleta de estado (`STATUS_COLORS`) que Resultados del TP1 y TP1 Aprobados.

In [ ]:
def failed_tp1_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning'], STATUS_COLORS['neutral']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultado en TP0 de los desaprobados de TP1',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que desaprobaron, descarta las columnas de notas (`Codigo`, `Pruebas`, `Informe`, `Nota final`) ya que no aportan nada al estar todos los registros desaprobados, y categoriza esos registros.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    failed_tp1_categories = ['TP0 Aprobado', 'TP0 Desaprobado', 'TP0 Sin Nota', 'TP0 N/A']

    # Filtra a los estudiantes que desaprobaron el TP1
    failed_tp1 = sheets[TP1_SHEET][sheets[TP1_SHEET]['Nota final'] == 'Desaprobado'].copy()

    # Descarta las columnas de notas: no aportan nada ya que todos los registros ya son desaprobados
    failed_tp1.drop(columns=['Codigo', 'Pruebas', 'Informe', 'Nota final'], errors='ignore', inplace=True)

    # Cruza con los datos de TP0 usando "Padron" y conservando "Aprobado". Si la hoja TP0 no
    # está disponible en esta planilla, se usa un DataFrame vacío: el merge no encuentra
    # coincidencias y todos los estudiantes caen en "TP0 N/A" más abajo.
    tp0_df = sheets.get(TP0_SHEET, pd.DataFrame(columns=['Padron', 'Aprobado']))[['Padron', 'Aprobado']]
    padrones_en_tp0 = set(tp0_df['Padron'])
    failed_tp1 = failed_tp1.merge(tp0_df, on='Padron', how='left')

    # Mapea la columna "Aprobado" a las categorías. Si el estudiante está en TP0 pero su
    # "Aprobado" no matchea ninguna categoría (corrección sin cerrar, valor no reconocido), es
    # "TP0 Sin Nota"; si directamente no aparece en TP0, es "TP0 N/A".
    failed_tp1['TP0_Final'] = failed_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'})
    en_tp0_sin_mapear = failed_tp1['TP0_Final'].isna() & failed_tp1['Padron'].isin(padrones_en_tp0)
    failed_tp1.loc[en_tp0_sin_mapear, 'TP0_Final'] = 'TP0 Sin Nota'
    failed_tp1['TP0_Final'] = failed_tp1['TP0_Final'].fillna('TP0 N/A')
    failed_tp1.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 4 categorías (incluso si el conteo es 0)
    failed_tp1_categories_count = failed_tp1['TP0_Final'].value_counts().reindex(failed_tp1_categories, fill_value=0)

    # Calcula el total de estudiantes
    failed_tp1_students_count = failed_tp1_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    display(failed_tp1)
    print(f"\n")
    failed_tp1_donut_chart(failed_tp1_categories, failed_tp1_categories_count, failed_tp1_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.5")

#### Intentos en TP1

Este módulo compara la cantidad de `intentos` de entrega entre quienes aprobaron y quienes desaprobaron el TP1, para ver si un mayor número de reintentos se asocia con el resultado final.

- Como `intentos` puede tomar muchos valores distintos con pocas ocurrencias cada uno, se agrupa en rangos fijos (`1-3`, `4-6`, `7-10`, `11-15`, `16-20`, `21+`) en vez de graficar un valor por barra.
- Al igual que en Notas del TP1, siempre se muestran los seis rangos aunque alguno tenga 0 alumnos, para que el gráfico no cambie de forma de un cuatrimestre a otro.
- El promedio que se muestra en la caja de Referencias se calcula sobre los valores reales de `intentos` (sin binnear), no sobre los rangos.

Como resultado, el módulo genera una figura con dos gráficos de barras — Aprobados y Desaprobados, con el mismo rango de bins en el eje X — más el total y el promedio de cada grupo en la caja de referencias.

##### Implementación

También delega el dibujo de cada barra en `_draw_count_bar`, esta vez sobre una figura de dos subplots (Aprobados y Desaprobados) que comparten el eje Y y el mismo rango de bins en el eje X, para que las alturas sean directamente comparables entre los dos grupos.

In [ ]:
def tp1_intentos_bar_chart(counts_aprobados, total_aprobados, mean_aprobados, counts_desaprobados, total_desaprobados, mean_desaprobados, fig_num):
  fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

  _draw_count_bar(axes[0], counts_aprobados, total_aprobados, xlabel='Intentos', title='Aprobados')
  _draw_count_bar(axes[1], counts_desaprobados, total_desaprobados, xlabel='Intentos', title='Desaprobados')
  axes[1].set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise al otro
  # panel; se recalcula una sola vez con el maximo real de ambos grupos.
  max_count = max(counts_aprobados.values.max(), counts_desaprobados.values.max())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Intentos en TP1',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  promedio_aprobados_txt = f"{mean_aprobados:.1f}" if total_aprobados > 0 else "N/A"
  promedio_desaprobados_txt = f"{mean_desaprobados:.1f}" if total_desaprobados > 0 else "N/A"
  legend_labels = [
      f"Aprobados: {total_aprobados} (prom. {promedio_aprobados_txt})",
      f"Desaprobados: {total_desaprobados} (prom. {promedio_desaprobados_txt})",
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.24, right=0.68, wspace=0.12)
  plt.show()

Esta sección separa a `TP1` en aprobados (mismo filtro que Notas del TP1) y desaprobados (mismo filtro que TP1 Desaprobados), convierte `intentos` a numérico, y cuenta cuántos alumnos de cada grupo caen en cada rango de intentos.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    intentos_bins = [0, 3, 6, 10, 15, 20, float('inf')]
    intentos_labels = ['1-3', '4-6', '7-10', '11-15', '16-20', '21+']

    # Aprobados: mismo filtro que tp1_notas
    tp1_intentos_aprobados = sheets[TP1_SHEET][~sheets[TP1_SHEET]['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])].copy()
    tp1_intentos_aprobados['intentos'] = pd.to_numeric(tp1_intentos_aprobados['intentos'].str.replace(',', '.', regex=False))
    tp1_intentos_aprobados_count = pd.cut(
        tp1_intentos_aprobados['intentos'], bins=intentos_bins, labels=intentos_labels
    ).value_counts().reindex(intentos_labels, fill_value=0)
    tp1_intentos_aprobados_total = len(tp1_intentos_aprobados)
    tp1_intentos_aprobados_mean = tp1_intentos_aprobados['intentos'].mean()

    # Desaprobados: mismo filtro que failed_tp1
    tp1_intentos_desaprobados = sheets[TP1_SHEET][sheets[TP1_SHEET]['Nota final'] == 'Desaprobado'].copy()
    tp1_intentos_desaprobados['intentos'] = pd.to_numeric(tp1_intentos_desaprobados['intentos'].str.replace(',', '.', regex=False))
    tp1_intentos_desaprobados_count = pd.cut(
        tp1_intentos_desaprobados['intentos'], bins=intentos_bins, labels=intentos_labels
    ).value_counts().reindex(intentos_labels, fill_value=0)
    tp1_intentos_desaprobados_total = len(tp1_intentos_desaprobados)
    tp1_intentos_desaprobados_mean = tp1_intentos_desaprobados['intentos'].mean()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_intentos_bar_chart(
        tp1_intentos_aprobados_count, tp1_intentos_aprobados_total, tp1_intentos_aprobados_mean,
        tp1_intentos_desaprobados_count, tp1_intentos_desaprobados_total, tp1_intentos_desaprobados_mean,
        fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.6"
    )

#### Estado en TP1 Desaprobados

Este módulo mira el `Estado` que el pipeline de corrección automática le asignó a cada desaprobado del TP1, para distinguir cuántos fallaron por un problema del propio pipeline (no compila, timeout) de cuántos sí llegaron a corregirse pero no alcanzaron la nota mínima.

- Normaliza los valores de `Estado`, que cambian de vocabulario entre esquemas de planilla (`ok`/`error` en 2025, `SUCCESS`/`FAILED`/`TIMEOUT` desde 2026), a tres categorías fijas: **OK**, **Timeout** y **Error**.
- Siempre muestra las tres categorías con su propia barra aunque alguna tenga 0 alumnos (por ejemplo, `Timeout` nunca aparece en el esquema de 2025) — un gráfico de torta esconde o directamente hace desaparecer las categorías en 0, y si casi todos los registros caen en un mismo estado, la torta queda prácticamente de un solo color y se ve poco informativa.
- A diferencia de los demás gráficos de barras de la notebook, que colorean según la magnitud de cada barra, acá el color es fijo por categoría — la paleta de estado (verde/ámbar/rojo) — porque comunica severidad, no una magnitud relativa.

Como resultado, el módulo genera un gráfico de barras con la proporción de cada estado entre los desaprobados.

##### Implementación

Esta función reutiliza `_draw_count_bar` (ver Utilidades) igual que Notas del TP1, pero le pasa una lista de colores fija en vez de dejar que calcule la rampa secuencial — la paleta de estado (`good`/`warning`/`critical`, verde/ámbar/rojo), ya que acá el color identifica cada estado y no una magnitud.

In [ ]:
def tp1_estado_bar_chart(categories_count, students_count, fig_num):
  plt.figure(figsize=(9, 5.5))
  ax = plt.gca()

  # Paleta de estado (good/warning/critical) en vez de la rampa secuencial de
  # _draw_count_bar: acá el color identifica cada estado, no una magnitud.
  status_colors = {'OK': '#0ca30c', 'Timeout': '#fab219', 'Error': '#d03b3b'}
  bar_colors = [status_colors[cat] for cat in categories_count.index]

  _draw_count_bar(ax, categories_count, students_count, xlabel='Estado', colors=bar_colors)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Estado en TP1 Desaprobados',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que Notas del TP1: las barras ya muestran cantidad y
  # porcentaje de cada estado, así que la caja solo necesita el total.
  legend_labels = [f"Total: {students_count} desaprobados"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.82, bottom=0.24, right=0.62)
  plt.show()

Esta sección filtra a los desaprobados del TP1 (mismo filtro que TP1 Desaprobados) y mapea su `Estado` a las tres categorías normalizadas, usando `_normalize_key` para que el mapeo no dependa de mayúsculas/minúsculas.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_estado_categories = ['OK', 'Timeout', 'Error']

    # El pipeline de corrección devuelve vocabularios distintos según el esquema de planilla:
    # 'ok'/'error' en 2025, 'SUCCESS'/'FAILED'/'TIMEOUT' desde 2026. Se normalizan a 3 categorías fijas.
    status_aliases = {'ok': 'OK', 'success': 'OK', 'error': 'Error', 'failed': 'Error', 'timeout': 'Timeout'}

    tp1_estado = sheets[TP1_SHEET][sheets[TP1_SHEET]['Nota final'] == 'Desaprobado'][['Padron', 'Estado']].copy()
    tp1_estado_crudo = tp1_estado['Estado']
    tp1_estado['Estado'] = tp1_estado_crudo.map(lambda s: status_aliases.get(_normalize_key(s)))
    _warn_unmapped(tp1_estado['Estado'], tp1_estado_crudo, "Estado en TP1 Desaprobados: columna 'Estado'")

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    tp1_estado_categories_count = tp1_estado['Estado'].value_counts().reindex(tp1_estado_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp1_estado_students_count = tp1_estado_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_estado_bar_chart(tp1_estado_categories_count, tp1_estado_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.7")

### LISTA

Estadísticas basadas en la hoja `LISTA`, el tercer trabajo práctico de la materia. Al igual que en TP1, el módulo Desaprobados cruza el resultado de LISTA con el de TP1, tratándolo como un antecedente opcional: si la hoja `TP1` no está disponible en esta planilla, o un estudiante puntual no tiene registro ahí, ese cruce cae en la categoría N/A (distinta de Sin Nota, ver el módulo) en vez de romper el análisis. A diferencia de TP1, LISTA no tiene un módulo "Aprobados" gemelo — ver Asimetría deliberada en NOTAS_INTERNAS.md.

#### Resultados de LISTA

Este módulo calcula la proporción de estudiantes que aprobaron, desaprobaron, o todavía no tienen nota en LISTA, replicando el mismo análisis de Resultados del TP1 (y RESULTADOS TP0) para el tercer trabajo práctico de la materia.

- Clasifica a cada estudiante presente en la hoja `LISTA` según su columna `Nota final`, en **Aprobado** (si tiene una nota numérica cargada), **Desaprobado** (si el valor es el string "Desaprobado"), o **Sin Nota** (si `Nota final` está vacío o marcado como "Reentrega" — corrección todavía abierta, o pendiente de una nueva entrega).

Como resultado, el módulo genera un donut para visualizar la proporción de cada categoría, con el total de alumnos en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo. Delega el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la paleta de estado fija (`STATUS_COLORS`), el mismo esquema que usan el resto de los módulos de esta sección y de TP1. Arma el título y la caja de Referencias; el propio donut ya muestra el total en su hueco central.

In [ ]:
def lista_results_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados de LISTA',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes de LISTA según su resultado en `Nota final`, con la misma lógica que Resultados del TP1.

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_results_categories = ['Aprobado', 'Desaprobado', 'Sin Nota']

    lista_results = sheets[LISTA_SHEET][['Padron', 'Nota final']].copy()

    # Clasifica según "Nota final": el string "Desaprobado", vacío o "Reentrega" (corrección todavía abierta o pendiente de una nueva entrega), o una nota numérica
    lista_results['Resultado'] = lista_results['Nota final'].apply(
        lambda nota: 'Desaprobado' if nota == 'Desaprobado' else ('Sin Nota' if nota in ('', 'Reentrega') else 'Aprobado')
    )
    lista_results.drop(columns=['Nota final'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    lista_results_categories_count = lista_results['Resultado'].value_counts().reindex(lista_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    lista_results_students_count = lista_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_results_donut_chart(lista_results_categories, lista_results_categories_count, lista_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['LISTA']}.1")

#### Notas de LISTA

Este módulo muestra la distribución de las notas numéricas obtenidas por quienes aprobaron LISTA, replicando el mismo análisis de Notas del TP1.

- Toma únicamente los registros de `LISTA` con una nota numérica cargada, descartando los desaprobados (`Nota final == "Desaprobado"`) y las correcciones todavía abiertas o en reentrega (`Nota final` vacío o "Reentrega").
- Siempre muestra **todas** las notas posibles del esquema vigente, aunque nadie haya sacado esa nota (con 0 alumnos), para que el gráfico no cambie de forma según qué notas salieron ese cuatrimestre.
- Cada barra se colorea con un degradé secuencial según su propia cantidad de alumnos (más clara = menos alumnos, más oscura = más alumnos), para que la nota más frecuente salte a la vista.

Como resultado, el módulo genera un gráfico de barras con el porcentaje y la cantidad de alumnos que sacó cada nota, más el total y el promedio en la caja de referencias.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de barras de este módulo. La rampa de color ordinal y el resto de la estética de cada barra viven en `_draw_count_bar` (ver Utilidades), igual que en Notas del TP1; esta función arma la figura, el título, y la caja de referencias con el total y el promedio.

In [ ]:
def lista_notas_bar_chart(notas_count, students_count, mean_nota, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, notas_count, students_count, xlabel='Nota')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas de LISTA',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas del TP1): sin categorías con
  # color que mostrar, los handles son invisibles (handlelength=0) y solo quedan las dos líneas de texto.
  promedio_txt = f"{mean_nota:.1f}" if students_count > 0 else "N/A"
  legend_labels = [f"Total: {students_count} aprobados", f"Promedio: {promedio_txt}"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.6)
  plt.show()

Esta sección filtra a los estudiantes que aprobaron LISTA, cuenta cuántos sacaron cada nota completando con 0 las notas del esquema vigente que nadie sacó, y calcula el promedio, reutilizando la misma detección de esquema (0-2 antes de 2026, 4-10 desde 2026) que Notas del TP1.

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_notas = sheets[LISTA_SHEET][['Padron', 'Nota final']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas)
    lista_notas = lista_notas[~lista_notas['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    # Google Sheets devuelve los decimales con coma (ej. "0,75") en planillas con configuración regional en español
    lista_notas['Nota'] = pd.to_numeric(lista_notas['Nota final'].str.replace(',', '.', regex=False))

    # Antes de 2026 las notas iban de 0 a 2 en intervalos de 0.25; desde 2026 van de 4 a 10 en intervalos de 1.
    # Como los rangos no se solapan, la nota máxima observada alcanza para inferir qué esquema aplica.
    lista_notas_pre_2026 = [round(i * 0.25, 2) for i in range(9)]
    lista_notas_2026_en_adelante = [float(i) for i in range(4, 11)]
    lista_max_nota = lista_notas['Nota'].max()
    lista_notas_posibles = lista_notas_2026_en_adelante if pd.isna(lista_max_nota) or lista_max_nota > 2 else lista_notas_pre_2026

    # Cuenta las ocurrencias por nota, completando con 0 las notas posibles que nadie sacó
    lista_notas_count = lista_notas['Nota'].value_counts().reindex(lista_notas_posibles, fill_value=0)

    # Calcula el total y el promedio de los aprobados
    lista_notas_students_count = lista_notas_count.sum()
    lista_notas_mean = lista_notas['Nota'].mean()

##### Resultado

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_notas_bar_chart(lista_notas_count, lista_notas_students_count, lista_notas_mean, fig_num=f"{SECTION_FIGURE_PREFIX['LISTA']}.2")

#### Notas por sección de LISTA

Este módulo desglosa la nota final de quienes aprobaron LISTA en sus tres componentes — `Codigo`, `Pruebas` e `Informe` — para ver si el rendimiento es parejo entre secciones o si alguna concentra las notas más bajas, replicando el mismo análisis de Notas por sección del TP1.

- Parte de la misma población que Notas de LISTA (aprobados con nota numérica cargada) y del mismo esquema de notas ya detectado para `Nota final`, ya que las tres secciones puntúan en la misma escala (0 a 2 antes de 2026, 4 a 10 desde 2026).
- Al igual que Notas de LISTA, siempre muestra todas las notas posibles del esquema vigente en cada sección, aunque nadie haya sacado esa nota, para que el gráfico no cambie de forma de un cuatrimestre a otro.

Como resultado, el módulo genera una figura con tres gráficos de barras — uno por sección, con el mismo eje Y para poder comparar alturas entre ellos — más el promedio de cada sección en la caja de referencias.

##### Implementación

Esta función arma una figura de tres subplots (uno por sección) y delega el dibujo de cada barra en `_draw_count_bar` (ver Utilidades), el mismo helper que usa Notas de LISTA — así evita repetir la lógica de la rampa de color, la grilla y los ejes tantas veces como secciones haya. Solo agrega el título de figura, el eje Y compartido, y una caja de referencias única con el promedio de cada sección.

In [ ]:
def lista_notas_por_seccion_bar_chart(secciones_counts, secciones_means, students_count, fig_num):
  secciones_labels = {'Codigo': 'Código', 'Pruebas': 'Pruebas', 'Informe': 'Informe'}

  fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), sharey=True)

  for ax, col in zip(axes, secciones_counts.keys()):
      _draw_count_bar(ax, secciones_counts[col], students_count, xlabel='Nota', title=secciones_labels[col])

  for ax in axes[1:]:
      ax.set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise a los
  # demas paneles; se recalcula una sola vez con el maximo real de las 3 secciones.
  max_count = max(counts.values.max() for counts in secciones_counts.values())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas por sección de LISTA',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas de LISTA), con un
  # promedio por sección en vez de uno solo.
  legend_labels = [f"Total: {students_count} aprobados"] + [
      f"Promedio {secciones_labels[col]}: {mean:.1f}" if students_count > 0 else f"Promedio {secciones_labels[col]}: N/A"
      for col, mean in secciones_means.items()
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.06, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.01,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.22, right=0.83, wspace=0.18)
  plt.show()

Esta sección filtra a los mismos aprobados que Notas de LISTA, convierte Codigo, Pruebas e Informe a numérico (mismo formato de coma decimal), y cuenta cuántos alumnos sacaron cada nota en cada sección, reutilizando el `lista_notas_posibles` ya calculado para `Nota final`.

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_secciones = sheets[LISTA_SHEET][['Padron', 'Nota final'] + ['Codigo', 'Pruebas', 'Informe']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas), mismo filtro que lista_notas
    lista_secciones = lista_secciones[~lista_secciones['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    lista_secciones_cols = ['Codigo', 'Pruebas', 'Informe']
    for col in lista_secciones_cols:
        # Mismo formato de coma decimal que el resto de las notas
        lista_secciones[col] = pd.to_numeric(lista_secciones[col].str.replace(',', '.', regex=False))

    # Reutiliza el esquema de notas ya detectado para "Nota final" (mismas escalas: 0-2 antes
    # de 2026, 4-10 desde 2026), ya que todas las secciones puntúan en la misma escala
    lista_secciones_counts = {
        col: lista_secciones[col].value_counts().reindex(lista_notas_posibles, fill_value=0)
        for col in lista_secciones_cols
    }
    lista_secciones_means = {col: lista_secciones[col].mean() for col in lista_secciones_cols}
    lista_secciones_students_count = len(lista_secciones)

##### Resultado

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_notas_por_seccion_bar_chart(lista_secciones_counts, lista_secciones_means, lista_secciones_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['LISTA']}.3")

#### Resultado en TP1 de los desaprobados de LISTA

Este módulo analiza el desempeño previo (opcional) en el TP1 de quienes desaprobaron LISTA, replicando el mismo análisis de TP1 Desaprobados. Esto puede darnos una idea de si desaprobar LISTA se relaciona con haber tenido dificultades ya en el trabajo anterior.

- Aísla a la población de estudiantes que desaprobó LISTA y, a partir de esos registros, cruza su resultado en el TP1 (derivado de su columna `Nota final`, con la misma lógica que Resultados de LISTA) para determinar si aprobaron, desaprobaron, todavía no tienen nota, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en `LISTA` pero ausentes en los registros de `TP1`, y los clasifica como *"No disponible"* (N/A). Esto es distinto de **Sin Nota**: un estudiante que sí está en `TP1` pero cuya `Nota final` todavía está vacía o en "Reentrega" — antes ambos casos se mezclaban bajo N/A; separarlos evita confundir "no entregó TP1" con "entregó TP1 pero todavía no tiene resultado".
- También asegura que el análisis siempre considere los cuatro resultados posibles de `TP1` (**Aprobado, Desaprobado, Sin Nota, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si la hoja `TP1` no está disponible en esta planilla (a diferencia del resto de las secciones, que se omiten enteras cuando falta la hoja que necesitan — ver `hoja_disponible` en Utilidades), este módulo igual corre: todos los estudiantes caen directamente en **TP1 N/A**, igual que si cada uno individualmente no tuviera registro en `TP1`.

Como resultado, el módulo devuelve un DataFrame con los datos de esos estudiantes, más un donut para una mejor visualización de los números, con el total de desaprobados en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo, delegando el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la misma paleta de estado (`STATUS_COLORS`) que Resultados de LISTA.

In [ ]:
def failed_lista_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning'], STATUS_COLORS['neutral']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultado en TP1 de los desaprobados de LISTA',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que desaprobaron, descarta las columnas de notas (`Codigo`, `Pruebas`, `Informe`, `Nota final`) ya que no aportan nada al estar todos los registros desaprobados, y categoriza esos registros según su TP1 (misma derivación desde `Nota final` que en Resultados de LISTA).

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    failed_lista_categories = ['TP1 Aprobado', 'TP1 Desaprobado', 'TP1 Sin Nota', 'TP1 N/A']

    # Filtra a los estudiantes que desaprobaron LISTA
    failed_lista = sheets[LISTA_SHEET][sheets[LISTA_SHEET]['Nota final'] == 'Desaprobado'].copy()

    # Descarta las columnas de notas: no aportan nada ya que todos los registros ya son desaprobados
    failed_lista.drop(columns=['Codigo', 'Pruebas', 'Informe', 'Nota final'], errors='ignore', inplace=True)

    # Deriva el resultado de TP1 a partir de su "Nota final" (misma lógica que Resultados de LISTA).
    # Si la hoja TP1 no está disponible en esta planilla, se usa un DataFrame vacío: el merge no
    # encuentra coincidencias y todos los estudiantes caen en "TP1 N/A" más abajo.
    lista_tp1_previo = sheets.get(TP1_SHEET, pd.DataFrame(columns=['Padron', 'Nota final']))[['Padron', 'Nota final']].copy()
    lista_tp1_previo['TP1_Final'] = lista_tp1_previo['Nota final'].apply(
        lambda nota: 'TP1 Desaprobado' if nota == 'Desaprobado' else ('TP1 Sin Nota' if nota in ('', 'Reentrega') else 'TP1 Aprobado')
    )

    failed_lista = failed_lista.merge(
        lista_tp1_previo[['Padron', 'TP1_Final']],
        on='Padron',
        how='left'
    )
    failed_lista['TP1_Final'] = failed_lista['TP1_Final'].fillna('TP1 N/A')

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 4 categorías (incluso si el conteo es 0)
    failed_lista_categories_count = failed_lista['TP1_Final'].value_counts().reindex(failed_lista_categories, fill_value=0)

    # Calcula el total de estudiantes
    failed_lista_students_count = failed_lista_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    display(failed_lista)
    print(f"\n")
    failed_lista_donut_chart(failed_lista_categories, failed_lista_categories_count, failed_lista_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['LISTA']}.4")

#### Intentos en LISTA

Este módulo compara la cantidad de `intentos` de entrega entre quienes aprobaron y quienes desaprobaron LISTA, replicando el mismo análisis de Intentos en TP1, para ver si un mayor número de reintentos se asocia con el resultado final.

- Como `intentos` puede tomar muchos valores distintos con pocas ocurrencias cada uno, se agrupa en rangos fijos (`1-3`, `4-6`, `7-10`, `11-15`, `16-20`, `21+`) en vez de graficar un valor por barra.
- Al igual que en Intentos en TP1, siempre se muestran los seis rangos aunque alguno tenga 0 alumnos, para que el gráfico no cambie de forma de un cuatrimestre a otro.
- El promedio que se muestra en la caja de Referencias se calcula sobre los valores reales de `intentos` (sin binnear), no sobre los rangos.

Como resultado, el módulo genera una figura con dos gráficos de barras — Aprobados y Desaprobados, con el mismo rango de bins en el eje X — más el total y el promedio de cada grupo en la caja de referencias.

##### Implementación

También delega el dibujo de cada barra en `_draw_count_bar`, esta vez sobre una figura de dos subplots (Aprobados y Desaprobados) que comparten el eje Y y el mismo rango de bins en el eje X, para que las alturas sean directamente comparables entre los dos grupos.

In [ ]:
def lista_intentos_bar_chart(counts_aprobados, total_aprobados, mean_aprobados, counts_desaprobados, total_desaprobados, mean_desaprobados, fig_num):
  fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

  _draw_count_bar(axes[0], counts_aprobados, total_aprobados, xlabel='Intentos', title='Aprobados')
  _draw_count_bar(axes[1], counts_desaprobados, total_desaprobados, xlabel='Intentos', title='Desaprobados')
  axes[1].set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise al otro
  # panel; se recalcula una sola vez con el maximo real de ambos grupos.
  max_count = max(counts_aprobados.values.max(), counts_desaprobados.values.max())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Intentos en LISTA',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  promedio_aprobados_txt = f"{mean_aprobados:.1f}" if total_aprobados > 0 else "N/A"
  promedio_desaprobados_txt = f"{mean_desaprobados:.1f}" if total_desaprobados > 0 else "N/A"
  legend_labels = [
      f"Aprobados: {total_aprobados} (prom. {promedio_aprobados_txt})",
      f"Desaprobados: {total_desaprobados} (prom. {promedio_desaprobados_txt})",
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.24, right=0.68, wspace=0.12)
  plt.show()

Esta sección separa LISTA en aprobados (mismo filtro que Notas de LISTA) y desaprobados (mismo filtro que LISTA Desaprobados), convierte `intentos` a numérico, y cuenta cuántos alumnos de cada grupo caen en cada rango de intentos.

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_intentos_bins = [0, 3, 6, 10, 15, 20, float('inf')]
    lista_intentos_labels = ['1-3', '4-6', '7-10', '11-15', '16-20', '21+']

    # Aprobados: mismo filtro que lista_notas
    lista_intentos_aprobados = sheets[LISTA_SHEET][~sheets[LISTA_SHEET]['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])].copy()
    lista_intentos_aprobados['intentos'] = pd.to_numeric(lista_intentos_aprobados['intentos'].str.replace(',', '.', regex=False))
    lista_intentos_aprobados_count = pd.cut(
        lista_intentos_aprobados['intentos'], bins=lista_intentos_bins, labels=lista_intentos_labels
    ).value_counts().reindex(lista_intentos_labels, fill_value=0)
    lista_intentos_aprobados_total = len(lista_intentos_aprobados)
    lista_intentos_aprobados_mean = lista_intentos_aprobados['intentos'].mean()

    # Desaprobados: mismo filtro que failed_lista
    lista_intentos_desaprobados = sheets[LISTA_SHEET][sheets[LISTA_SHEET]['Nota final'] == 'Desaprobado'].copy()
    lista_intentos_desaprobados['intentos'] = pd.to_numeric(lista_intentos_desaprobados['intentos'].str.replace(',', '.', regex=False))
    lista_intentos_desaprobados_count = pd.cut(
        lista_intentos_desaprobados['intentos'], bins=lista_intentos_bins, labels=lista_intentos_labels
    ).value_counts().reindex(lista_intentos_labels, fill_value=0)
    lista_intentos_desaprobados_total = len(lista_intentos_desaprobados)
    lista_intentos_desaprobados_mean = lista_intentos_desaprobados['intentos'].mean()

##### Resultado

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_intentos_bar_chart(
        lista_intentos_aprobados_count, lista_intentos_aprobados_total, lista_intentos_aprobados_mean,
        lista_intentos_desaprobados_count, lista_intentos_desaprobados_total, lista_intentos_desaprobados_mean,
        fig_num=f"{SECTION_FIGURE_PREFIX['LISTA']}.5"
    )

#### Estado en LISTA Desaprobados

Este módulo mira el `Estado` que el pipeline de corrección automática le asignó a cada desaprobado de LISTA, replicando el mismo análisis de Estado en TP1 Desaprobados, para distinguir cuántos fallaron por un problema del propio pipeline (no compila, timeout) de cuántos sí llegaron a corregirse pero no alcanzaron la nota mínima.

- Normaliza los valores de `Estado`, que cambian de vocabulario entre esquemas de planilla (`ok`/`error` en 2025, `SUCCESS`/`FAILED`/`TIMEOUT` desde 2026), a tres categorías fijas: **OK**, **Timeout** y **Error**.
- Siempre muestra las tres categorías con su propia barra aunque alguna tenga 0 alumnos, por la misma razón que en Estado en TP1 Desaprobados: un gráfico de torta esconde o directamente hace desaparecer las categorías en 0.
- A diferencia de los demás gráficos de barras de la notebook, que colorean según la magnitud de cada barra, acá el color es fijo por categoría — la paleta de estado (verde/ámbar/rojo) — porque comunica severidad, no una magnitud relativa.

Como resultado, el módulo genera un gráfico de barras con la proporción de cada estado entre los desaprobados.

##### Implementación

Esta función reutiliza `_draw_count_bar` (ver Utilidades) igual que Notas de LISTA, pero le pasa una lista de colores fija en vez de dejar que calcule la rampa secuencial — la paleta de estado (`good`/`warning`/`critical`, verde/ámbar/rojo), ya que acá el color identifica cada estado y no una magnitud.

In [ ]:
def lista_estado_bar_chart(categories_count, students_count, fig_num):
  plt.figure(figsize=(9, 5.5))
  ax = plt.gca()

  # Paleta de estado (good/warning/critical) en vez de la rampa secuencial de
  # _draw_count_bar: acá el color identifica cada estado, no una magnitud.
  status_colors = {'OK': '#0ca30c', 'Timeout': '#fab219', 'Error': '#d03b3b'}
  bar_colors = [status_colors[cat] for cat in categories_count.index]

  _draw_count_bar(ax, categories_count, students_count, xlabel='Estado', colors=bar_colors)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Estado en LISTA Desaprobados',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que Notas de LISTA: las barras ya muestran cantidad y
  # porcentaje de cada estado, así que la caja solo necesita el total.
  legend_labels = [f"Total: {students_count} desaprobados"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.82, bottom=0.24, right=0.62)
  plt.show()

Esta sección filtra a los desaprobados de LISTA (mismo filtro que LISTA Desaprobados) y mapea su `Estado` a las tres categorías normalizadas, usando `_normalize_key` para que el mapeo no dependa de mayúsculas/minúsculas.

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_estado_categories = ['OK', 'Timeout', 'Error']

    # El pipeline de corrección devuelve vocabularios distintos según el esquema de planilla:
    # 'ok'/'error' en 2025, 'SUCCESS'/'FAILED'/'TIMEOUT' desde 2026. Se normalizan a 3 categorías fijas.
    lista_status_aliases = {'ok': 'OK', 'success': 'OK', 'error': 'Error', 'failed': 'Error', 'timeout': 'Timeout'}

    lista_estado = sheets[LISTA_SHEET][sheets[LISTA_SHEET]['Nota final'] == 'Desaprobado'][['Padron', 'Estado']].copy()
    lista_estado_crudo = lista_estado['Estado']
    lista_estado['Estado'] = lista_estado_crudo.map(lambda s: lista_status_aliases.get(_normalize_key(s)))
    _warn_unmapped(lista_estado['Estado'], lista_estado_crudo, "Estado en LISTA Desaprobados: columna 'Estado'")

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    lista_estado_categories_count = lista_estado['Estado'].value_counts().reindex(lista_estado_categories, fill_value=0)

    # Calcula el total de estudiantes
    lista_estado_students_count = lista_estado_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[LISTA_SHEET]:
    lista_estado_bar_chart(lista_estado_categories_count, lista_estado_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['LISTA']}.6")

### ABB

Estadísticas basadas en la hoja `ABB`, el cuarto trabajo práctico de la materia. Al igual que en LISTA, el módulo Desaprobados cruza el resultado de ABB con el de LISTA, tratándolo como un antecedente opcional. A diferencia del resto de las secciones de **Estadísticas**, que arrancan abiertas, esta sección arranca colapsada por default: `ABB` dejó de tomarse como trabajo práctico desde 2026 (ver `hoja_disponible` en Utilidades), así que sus estadísticas solo aplican a cuatrimestres anteriores.

#### Resultados de ABB

Este módulo calcula la proporción de estudiantes que aprobaron, desaprobaron, o todavía no tienen nota en ABB, replicando el mismo análisis de Resultados del TP1 (y RESULTADOS TP0) para el cuarto trabajo práctico de la materia.

- Clasifica a cada estudiante presente en la hoja `ABB` según su columna `Nota final`, en **Aprobado** (si tiene una nota numérica cargada), **Desaprobado** (si el valor es el string "Desaprobado"), o **Sin Nota** (si `Nota final` está vacío o marcado como "Reentrega" — corrección todavía abierta, o pendiente de una nueva entrega).

Como resultado, el módulo genera un donut para visualizar la proporción de cada categoría, con el total de alumnos en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo. Delega el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la paleta de estado fija (`STATUS_COLORS`), el mismo esquema que usan el resto de los módulos de esta sección y de TP1. Arma el título y la caja de Referencias; el propio donut ya muestra el total en su hueco central.

In [ ]:
def abb_results_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados de ABB',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes de ABB según su resultado en `Nota final`, con la misma lógica que Resultados del TP1.

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_results_categories = ['Aprobado', 'Desaprobado', 'Sin Nota']

    abb_results = sheets[ABB_SHEET][['Padron', 'Nota final']].copy()

    # Clasifica según "Nota final": el string "Desaprobado", vacío o "Reentrega" (corrección todavía abierta o pendiente de una nueva entrega), o una nota numérica
    abb_results['Resultado'] = abb_results['Nota final'].apply(
        lambda nota: 'Desaprobado' if nota == 'Desaprobado' else ('Sin Nota' if nota in ('', 'Reentrega') else 'Aprobado')
    )
    abb_results.drop(columns=['Nota final'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    abb_results_categories_count = abb_results['Resultado'].value_counts().reindex(abb_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    abb_results_students_count = abb_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_results_donut_chart(abb_results_categories, abb_results_categories_count, abb_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['ABB']}.1")

#### Notas de ABB

Este módulo muestra la distribución de las notas numéricas obtenidas por quienes aprobaron ABB, replicando el mismo análisis de Notas del TP1.

- Toma únicamente los registros de `ABB` con una nota numérica cargada, descartando los desaprobados (`Nota final == "Desaprobado"`) y las correcciones todavía abiertas o en reentrega (`Nota final` vacío o "Reentrega").
- Siempre muestra **todas** las notas posibles del esquema vigente, aunque nadie haya sacado esa nota (con 0 alumnos), para que el gráfico no cambie de forma según qué notas salieron ese cuatrimestre.
- Cada barra se colorea con un degradé secuencial según su propia cantidad de alumnos (más clara = menos alumnos, más oscura = más alumnos), para que la nota más frecuente salte a la vista.

Como resultado, el módulo genera un gráfico de barras con el porcentaje y la cantidad de alumnos que sacó cada nota, más el total y el promedio en la caja de referencias.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de barras de este módulo. La rampa de color ordinal y el resto de la estética de cada barra viven en `_draw_count_bar` (ver Utilidades), igual que en Notas del TP1; esta función arma la figura, el título, y la caja de referencias con el total y el promedio.

In [ ]:
def abb_notas_bar_chart(notas_count, students_count, mean_nota, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, notas_count, students_count, xlabel='Nota')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas de ABB',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas del TP1): sin categorías con
  # color que mostrar, los handles son invisibles (handlelength=0) y solo quedan las dos líneas de texto.
  promedio_txt = f"{mean_nota:.1f}" if students_count > 0 else "N/A"
  legend_labels = [f"Total: {students_count} aprobados", f"Promedio: {promedio_txt}"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.6)
  plt.show()

Esta sección filtra a los estudiantes que aprobaron ABB, cuenta cuántos sacaron cada nota completando con 0 las notas del esquema vigente que nadie sacó, y calcula el promedio, reutilizando la misma detección de esquema (0-2 antes de 2026, 4-10 desde 2026) que Notas del TP1.

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_notas = sheets[ABB_SHEET][['Padron', 'Nota final']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas)
    abb_notas = abb_notas[~abb_notas['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    # Google Sheets devuelve los decimales con coma (ej. "0,75") en planillas con configuración regional en español
    abb_notas['Nota'] = pd.to_numeric(abb_notas['Nota final'].str.replace(',', '.', regex=False))

    # Antes de 2026 las notas iban de 0 a 2 en intervalos de 0.25; desde 2026 van de 4 a 10 en intervalos de 1.
    # Como los rangos no se solapan, la nota máxima observada alcanza para inferir qué esquema aplica.
    abb_notas_pre_2026 = [round(i * 0.25, 2) for i in range(9)]
    abb_notas_2026_en_adelante = [float(i) for i in range(4, 11)]
    abb_max_nota = abb_notas['Nota'].max()
    abb_notas_posibles = abb_notas_2026_en_adelante if pd.isna(abb_max_nota) or abb_max_nota > 2 else abb_notas_pre_2026

    # Cuenta las ocurrencias por nota, completando con 0 las notas posibles que nadie sacó
    abb_notas_count = abb_notas['Nota'].value_counts().reindex(abb_notas_posibles, fill_value=0)

    # Calcula el total y el promedio de los aprobados
    abb_notas_students_count = abb_notas_count.sum()
    abb_notas_mean = abb_notas['Nota'].mean()

##### Resultado

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_notas_bar_chart(abb_notas_count, abb_notas_students_count, abb_notas_mean, fig_num=f"{SECTION_FIGURE_PREFIX['ABB']}.2")

#### Notas por sección de ABB

Este módulo desglosa la nota final de quienes aprobaron ABB en sus tres componentes — `Codigo`, `Pruebas` e `Informe` — para ver si el rendimiento es parejo entre secciones o si alguna concentra las notas más bajas, replicando el mismo análisis de Notas por sección del TP1.

- Parte de la misma población que Notas de ABB (aprobados con nota numérica cargada) y del mismo esquema de notas ya detectado para `Nota final`, ya que las tres secciones puntúan en la misma escala (0 a 2 antes de 2026, 4 a 10 desde 2026).
- Al igual que Notas de ABB, siempre muestra todas las notas posibles del esquema vigente en cada sección, aunque nadie haya sacado esa nota, para que el gráfico no cambie de forma de un cuatrimestre a otro.

Como resultado, el módulo genera una figura con tres gráficos de barras — uno por sección, con el mismo eje Y para poder comparar alturas entre ellos — más el promedio de cada sección en la caja de referencias.

##### Implementación

Esta función arma una figura de tres subplots (uno por sección) y delega el dibujo de cada barra en `_draw_count_bar` (ver Utilidades), el mismo helper que usa Notas de ABB — así evita repetir la lógica de la rampa de color, la grilla y los ejes tantas veces como secciones haya. Solo agrega el título de figura, el eje Y compartido, y una caja de referencias única con el promedio de cada sección.

In [ ]:
def abb_notas_por_seccion_bar_chart(secciones_counts, secciones_means, students_count, fig_num):
  secciones_labels = {'Codigo': 'Código', 'Pruebas': 'Pruebas', 'Informe': 'Informe'}

  fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), sharey=True)

  for ax, col in zip(axes, secciones_counts.keys()):
      _draw_count_bar(ax, secciones_counts[col], students_count, xlabel='Nota', title=secciones_labels[col])

  for ax in axes[1:]:
      ax.set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise a los
  # demas paneles; se recalcula una sola vez con el maximo real de las 3 secciones.
  max_count = max(counts.values.max() for counts in secciones_counts.values())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas por sección de ABB',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas de ABB), con un
  # promedio por sección en vez de uno solo.
  legend_labels = [f"Total: {students_count} aprobados"] + [
      f"Promedio {secciones_labels[col]}: {mean:.1f}" if students_count > 0 else f"Promedio {secciones_labels[col]}: N/A"
      for col, mean in secciones_means.items()
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.06, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.01,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.22, right=0.83, wspace=0.18)
  plt.show()

Esta sección filtra a los mismos aprobados que Notas de ABB, convierte Codigo, Pruebas e Informe a numérico (mismo formato de coma decimal), y cuenta cuántos alumnos sacaron cada nota en cada sección, reutilizando el `abb_notas_posibles` ya calculado para `Nota final`.

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_secciones = sheets[ABB_SHEET][['Padron', 'Nota final'] + ['Codigo', 'Pruebas', 'Informe']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas), mismo filtro que abb_notas
    abb_secciones = abb_secciones[~abb_secciones['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    abb_secciones_cols = ['Codigo', 'Pruebas', 'Informe']
    for col in abb_secciones_cols:
        # Mismo formato de coma decimal que el resto de las notas
        abb_secciones[col] = pd.to_numeric(abb_secciones[col].str.replace(',', '.', regex=False))

    # Reutiliza el esquema de notas ya detectado para "Nota final" (mismas escalas: 0-2 antes
    # de 2026, 4-10 desde 2026), ya que todas las secciones puntúan en la misma escala
    abb_secciones_counts = {
        col: abb_secciones[col].value_counts().reindex(abb_notas_posibles, fill_value=0)
        for col in abb_secciones_cols
    }
    abb_secciones_means = {col: abb_secciones[col].mean() for col in abb_secciones_cols}
    abb_secciones_students_count = len(abb_secciones)

##### Resultado

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_notas_por_seccion_bar_chart(abb_secciones_counts, abb_secciones_means, abb_secciones_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['ABB']}.3")

#### Resultado en LISTA de los desaprobados de ABB

Este módulo analiza el desempeño previo (opcional) en LISTA de quienes desaprobaron ABB, replicando el mismo análisis de TP1 Desaprobados. Esto puede darnos una idea de si desaprobar ABB se relaciona con haber tenido dificultades ya en el trabajo anterior.

- Aísla a la población de estudiantes que desaprobó ABB y, a partir de esos registros, cruza su resultado en LISTA (derivado de su columna `Nota final`, con la misma lógica que Resultados de ABB) para determinar si aprobaron, desaprobaron, todavía no tienen nota, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en `ABB` pero ausentes en los registros de `LISTA`, y los clasifica como *"No disponible"* (N/A). Esto es distinto de **Sin Nota**: un estudiante que sí está en `LISTA` pero cuya `Nota final` todavía está vacía o en "Reentrega" — antes ambos casos se mezclaban bajo N/A; separarlos evita confundir "no entregó LISTA" con "entregó LISTA pero todavía no tiene resultado".
- También asegura que el análisis siempre considere los cuatro resultados posibles de `LISTA` (**Aprobado, Desaprobado, Sin Nota, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si la hoja `LISTA` no está disponible en esta planilla (a diferencia del resto de las secciones, que se omiten enteras cuando falta la hoja que necesitan — ver `hoja_disponible` en Utilidades), este módulo igual corre: todos los estudiantes caen directamente en **LISTA N/A**, igual que si cada uno individualmente no tuviera registro en `LISTA`.

Como resultado, el módulo devuelve un DataFrame con los datos de esos estudiantes, más un donut para una mejor visualización de los números, con el total de desaprobados en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo, delegando el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la misma paleta de estado (`STATUS_COLORS`) que Resultados de ABB.

In [ ]:
def failed_abb_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning'], STATUS_COLORS['neutral']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultado en LISTA de los desaprobados de ABB',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que desaprobaron, descarta las columnas de notas (`Codigo`, `Pruebas`, `Informe`, `Nota final`) ya que no aportan nada al estar todos los registros desaprobados, y categoriza esos registros según su LISTA (misma derivación desde `Nota final` que en Resultados de ABB).

In [ ]:
if hoja_disponible[ABB_SHEET]:
    failed_abb_categories = ['LISTA Aprobado', 'LISTA Desaprobado', 'LISTA Sin Nota', 'LISTA N/A']

    # Filtra a los estudiantes que desaprobaron ABB
    failed_abb = sheets[ABB_SHEET][sheets[ABB_SHEET]['Nota final'] == 'Desaprobado'].copy()

    # Descarta las columnas de notas: no aportan nada ya que todos los registros ya son desaprobados
    failed_abb.drop(columns=['Codigo', 'Pruebas', 'Informe', 'Nota final'], errors='ignore', inplace=True)

    # Deriva el resultado de LISTA a partir de su "Nota final" (misma lógica que Resultados de ABB).
    # Si la hoja LISTA no está disponible en esta planilla, se usa un DataFrame vacío: el merge no
    # encuentra coincidencias y todos los estudiantes caen en "LISTA N/A" más abajo.
    abb_lista_previo = sheets.get(LISTA_SHEET, pd.DataFrame(columns=['Padron', 'Nota final']))[['Padron', 'Nota final']].copy()
    abb_lista_previo['LISTA_Final'] = abb_lista_previo['Nota final'].apply(
        lambda nota: 'LISTA Desaprobado' if nota == 'Desaprobado' else ('LISTA Sin Nota' if nota in ('', 'Reentrega') else 'LISTA Aprobado')
    )

    failed_abb = failed_abb.merge(
        abb_lista_previo[['Padron', 'LISTA_Final']],
        on='Padron',
        how='left'
    )
    failed_abb['LISTA_Final'] = failed_abb['LISTA_Final'].fillna('LISTA N/A')

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 4 categorías (incluso si el conteo es 0)
    failed_abb_categories_count = failed_abb['LISTA_Final'].value_counts().reindex(failed_abb_categories, fill_value=0)

    # Calcula el total de estudiantes
    failed_abb_students_count = failed_abb_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[ABB_SHEET]:
    display(failed_abb)
    print(f"\n")
    failed_abb_donut_chart(failed_abb_categories, failed_abb_categories_count, failed_abb_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['ABB']}.4")

#### Intentos en ABB

Este módulo compara la cantidad de `intentos` de entrega entre quienes aprobaron y quienes desaprobaron ABB, replicando el mismo análisis de Intentos en TP1, para ver si un mayor número de reintentos se asocia con el resultado final.

- Como `intentos` puede tomar muchos valores distintos con pocas ocurrencias cada uno, se agrupa en rangos fijos (`1-3`, `4-6`, `7-10`, `11-15`, `16-20`, `21+`) en vez de graficar un valor por barra.
- Al igual que en Intentos en TP1, siempre se muestran los seis rangos aunque alguno tenga 0 alumnos, para que el gráfico no cambie de forma de un cuatrimestre a otro.
- El promedio que se muestra en la caja de Referencias se calcula sobre los valores reales de `intentos` (sin binnear), no sobre los rangos.

Como resultado, el módulo genera una figura con dos gráficos de barras — Aprobados y Desaprobados, con el mismo rango de bins en el eje X — más el total y el promedio de cada grupo en la caja de referencias.

##### Implementación

También delega el dibujo de cada barra en `_draw_count_bar`, esta vez sobre una figura de dos subplots (Aprobados y Desaprobados) que comparten el eje Y y el mismo rango de bins en el eje X, para que las alturas sean directamente comparables entre los dos grupos.

In [ ]:
def abb_intentos_bar_chart(counts_aprobados, total_aprobados, mean_aprobados, counts_desaprobados, total_desaprobados, mean_desaprobados, fig_num):
  fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

  _draw_count_bar(axes[0], counts_aprobados, total_aprobados, xlabel='Intentos', title='Aprobados')
  _draw_count_bar(axes[1], counts_desaprobados, total_desaprobados, xlabel='Intentos', title='Desaprobados')
  axes[1].set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise al otro
  # panel; se recalcula una sola vez con el maximo real de ambos grupos.
  max_count = max(counts_aprobados.values.max(), counts_desaprobados.values.max())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Intentos en ABB',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  promedio_aprobados_txt = f"{mean_aprobados:.1f}" if total_aprobados > 0 else "N/A"
  promedio_desaprobados_txt = f"{mean_desaprobados:.1f}" if total_desaprobados > 0 else "N/A"
  legend_labels = [
      f"Aprobados: {total_aprobados} (prom. {promedio_aprobados_txt})",
      f"Desaprobados: {total_desaprobados} (prom. {promedio_desaprobados_txt})",
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.24, right=0.68, wspace=0.12)
  plt.show()

Esta sección separa ABB en aprobados (mismo filtro que Notas de ABB) y desaprobados (mismo filtro que ABB Desaprobados), convierte `intentos` a numérico, y cuenta cuántos alumnos de cada grupo caen en cada rango de intentos.

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_intentos_bins = [0, 3, 6, 10, 15, 20, float('inf')]
    abb_intentos_labels = ['1-3', '4-6', '7-10', '11-15', '16-20', '21+']

    # Aprobados: mismo filtro que abb_notas
    abb_intentos_aprobados = sheets[ABB_SHEET][~sheets[ABB_SHEET]['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])].copy()
    abb_intentos_aprobados['intentos'] = pd.to_numeric(abb_intentos_aprobados['intentos'].str.replace(',', '.', regex=False))
    abb_intentos_aprobados_count = pd.cut(
        abb_intentos_aprobados['intentos'], bins=abb_intentos_bins, labels=abb_intentos_labels
    ).value_counts().reindex(abb_intentos_labels, fill_value=0)
    abb_intentos_aprobados_total = len(abb_intentos_aprobados)
    abb_intentos_aprobados_mean = abb_intentos_aprobados['intentos'].mean()

    # Desaprobados: mismo filtro que failed_abb
    abb_intentos_desaprobados = sheets[ABB_SHEET][sheets[ABB_SHEET]['Nota final'] == 'Desaprobado'].copy()
    abb_intentos_desaprobados['intentos'] = pd.to_numeric(abb_intentos_desaprobados['intentos'].str.replace(',', '.', regex=False))
    abb_intentos_desaprobados_count = pd.cut(
        abb_intentos_desaprobados['intentos'], bins=abb_intentos_bins, labels=abb_intentos_labels
    ).value_counts().reindex(abb_intentos_labels, fill_value=0)
    abb_intentos_desaprobados_total = len(abb_intentos_desaprobados)
    abb_intentos_desaprobados_mean = abb_intentos_desaprobados['intentos'].mean()

##### Resultado

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_intentos_bar_chart(
        abb_intentos_aprobados_count, abb_intentos_aprobados_total, abb_intentos_aprobados_mean,
        abb_intentos_desaprobados_count, abb_intentos_desaprobados_total, abb_intentos_desaprobados_mean,
        fig_num=f"{SECTION_FIGURE_PREFIX['ABB']}.5"
    )

#### Estado en ABB Desaprobados

Este módulo mira el `Estado` que el pipeline de corrección automática le asignó a cada desaprobado de ABB, replicando el mismo análisis de Estado en TP1 Desaprobados, para distinguir cuántos fallaron por un problema del propio pipeline (no compila, timeout) de cuántos sí llegaron a corregirse pero no alcanzaron la nota mínima.

- Normaliza los valores de `Estado`, que cambian de vocabulario entre esquemas de planilla (`ok`/`error` en 2025, `SUCCESS`/`FAILED`/`TIMEOUT` desde 2026), a tres categorías fijas: **OK**, **Timeout** y **Error**.
- Siempre muestra las tres categorías con su propia barra aunque alguna tenga 0 alumnos, por la misma razón que en Estado en TP1 Desaprobados: un gráfico de torta esconde o directamente hace desaparecer las categorías en 0.
- A diferencia de los demás gráficos de barras de la notebook, que colorean según la magnitud de cada barra, acá el color es fijo por categoría — la paleta de estado (verde/ámbar/rojo) — porque comunica severidad, no una magnitud relativa.

Como resultado, el módulo genera un gráfico de barras con la proporción de cada estado entre los desaprobados.

##### Implementación

Esta función reutiliza `_draw_count_bar` (ver Utilidades) igual que Notas de ABB, pero le pasa una lista de colores fija en vez de dejar que calcule la rampa secuencial — la paleta de estado (`good`/`warning`/`critical`, verde/ámbar/rojo), ya que acá el color identifica cada estado y no una magnitud.

In [ ]:
def abb_estado_bar_chart(categories_count, students_count, fig_num):
  plt.figure(figsize=(9, 5.5))
  ax = plt.gca()

  # Paleta de estado (good/warning/critical) en vez de la rampa secuencial de
  # _draw_count_bar: acá el color identifica cada estado, no una magnitud.
  status_colors = {'OK': '#0ca30c', 'Timeout': '#fab219', 'Error': '#d03b3b'}
  bar_colors = [status_colors[cat] for cat in categories_count.index]

  _draw_count_bar(ax, categories_count, students_count, xlabel='Estado', colors=bar_colors)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Estado en ABB Desaprobados',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que Notas de ABB: las barras ya muestran cantidad y
  # porcentaje de cada estado, así que la caja solo necesita el total.
  legend_labels = [f"Total: {students_count} desaprobados"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.82, bottom=0.24, right=0.62)
  plt.show()

Esta sección filtra a los desaprobados de ABB (mismo filtro que ABB Desaprobados) y mapea su `Estado` a las tres categorías normalizadas, usando `_normalize_key` para que el mapeo no dependa de mayúsculas/minúsculas.

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_estado_categories = ['OK', 'Timeout', 'Error']

    # El pipeline de corrección devuelve vocabularios distintos según el esquema de planilla:
    # 'ok'/'error' en 2025, 'SUCCESS'/'FAILED'/'TIMEOUT' desde 2026. Se normalizan a 3 categorías fijas.
    abb_status_aliases = {'ok': 'OK', 'success': 'OK', 'error': 'Error', 'failed': 'Error', 'timeout': 'Timeout'}

    abb_estado = sheets[ABB_SHEET][sheets[ABB_SHEET]['Nota final'] == 'Desaprobado'][['Padron', 'Estado']].copy()
    abb_estado_crudo = abb_estado['Estado']
    abb_estado['Estado'] = abb_estado_crudo.map(lambda s: abb_status_aliases.get(_normalize_key(s)))
    _warn_unmapped(abb_estado['Estado'], abb_estado_crudo, "Estado en ABB Desaprobados: columna 'Estado'")

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    abb_estado_categories_count = abb_estado['Estado'].value_counts().reindex(abb_estado_categories, fill_value=0)

    # Calcula el total de estudiantes
    abb_estado_students_count = abb_estado_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[ABB_SHEET]:
    abb_estado_bar_chart(abb_estado_categories_count, abb_estado_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['ABB']}.6")

### HASH

Estadísticas basadas en la hoja `HASH`, el quinto trabajo práctico de la materia. El módulo Desaprobados cruza el resultado de HASH con el de su TP anterior — pero, a diferencia del resto de las secciones, ese anterior no es fijo: es `ABB` si esa hoja está disponible en esta planilla, o `LISTA` si no lo está. Esto hace que la desaparición de `ABB` desde 2026 no degrade a todos los estudiantes a N/A: el módulo simplemente salta el hueco y compara contra `LISTA` en su lugar.

#### Resultados de HASH

Este módulo calcula la proporción de estudiantes que aprobaron, desaprobaron, o todavía no tienen nota en HASH, replicando el mismo análisis de Resultados del TP1 (y RESULTADOS TP0) para el quinto trabajo práctico de la materia.

- Clasifica a cada estudiante presente en la hoja `HASH` según su columna `Nota final`, en **Aprobado** (si tiene una nota numérica cargada), **Desaprobado** (si el valor es el string "Desaprobado"), o **Sin Nota** (si `Nota final` está vacío o marcado como "Reentrega" — corrección todavía abierta, o pendiente de una nueva entrega).

Como resultado, el módulo genera un donut para visualizar la proporción de cada categoría, con el total de alumnos en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo. Delega el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la paleta de estado fija (`STATUS_COLORS`), el mismo esquema que usan el resto de los módulos de esta sección y de TP1. Arma el título y la caja de Referencias; el propio donut ya muestra el total en su hueco central.

In [ ]:
def hash_results_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados de HASH',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes de HASH según su resultado en `Nota final`, con la misma lógica que Resultados del TP1.

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_results_categories = ['Aprobado', 'Desaprobado', 'Sin Nota']

    hash_results = sheets[HASH_SHEET][['Padron', 'Nota final']].copy()

    # Clasifica según "Nota final": el string "Desaprobado", vacío o "Reentrega" (corrección todavía abierta o pendiente de una nueva entrega), o una nota numérica
    hash_results['Resultado'] = hash_results['Nota final'].apply(
        lambda nota: 'Desaprobado' if nota == 'Desaprobado' else ('Sin Nota' if nota in ('', 'Reentrega') else 'Aprobado')
    )
    hash_results.drop(columns=['Nota final'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    hash_results_categories_count = hash_results['Resultado'].value_counts().reindex(hash_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    hash_results_students_count = hash_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_results_donut_chart(hash_results_categories, hash_results_categories_count, hash_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['HASH']}.1")

#### Notas de HASH

Este módulo muestra la distribución de las notas numéricas obtenidas por quienes aprobaron HASH, replicando el mismo análisis de Notas del TP1.

- Toma únicamente los registros de `HASH` con una nota numérica cargada, descartando los desaprobados (`Nota final == "Desaprobado"`) y las correcciones todavía abiertas o en reentrega (`Nota final` vacío o "Reentrega").
- Siempre muestra **todas** las notas posibles del esquema vigente, aunque nadie haya sacado esa nota (con 0 alumnos), para que el gráfico no cambie de forma según qué notas salieron ese cuatrimestre.
- Cada barra se colorea con un degradé secuencial según su propia cantidad de alumnos (más clara = menos alumnos, más oscura = más alumnos), para que la nota más frecuente salte a la vista.

Como resultado, el módulo genera un gráfico de barras con el porcentaje y la cantidad de alumnos que sacó cada nota, más el total y el promedio en la caja de referencias.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de barras de este módulo. La rampa de color ordinal y el resto de la estética de cada barra viven en `_draw_count_bar` (ver Utilidades), igual que en Notas del TP1; esta función arma la figura, el título, y la caja de referencias con el total y el promedio.

In [ ]:
def hash_notas_bar_chart(notas_count, students_count, mean_nota, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, notas_count, students_count, xlabel='Nota')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas de HASH',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas del TP1): sin categorías con
  # color que mostrar, los handles son invisibles (handlelength=0) y solo quedan las dos líneas de texto.
  promedio_txt = f"{mean_nota:.1f}" if students_count > 0 else "N/A"
  legend_labels = [f"Total: {students_count} aprobados", f"Promedio: {promedio_txt}"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.6)
  plt.show()

Esta sección filtra a los estudiantes que aprobaron HASH, cuenta cuántos sacaron cada nota completando con 0 las notas del esquema vigente que nadie sacó, y calcula el promedio, reutilizando la misma detección de esquema (0-2 antes de 2026, 4-10 desde 2026) que Notas del TP1.

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_notas = sheets[HASH_SHEET][['Padron', 'Nota final']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas)
    hash_notas = hash_notas[~hash_notas['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    # Google Sheets devuelve los decimales con coma (ej. "0,75") en planillas con configuración regional en español
    hash_notas['Nota'] = pd.to_numeric(hash_notas['Nota final'].str.replace(',', '.', regex=False))

    # Antes de 2026 las notas iban de 0 a 2 en intervalos de 0.25; desde 2026 van de 4 a 10 en intervalos de 1.
    # Como los rangos no se solapan, la nota máxima observada alcanza para inferir qué esquema aplica.
    hash_notas_pre_2026 = [round(i * 0.25, 2) for i in range(9)]
    hash_notas_2026_en_adelante = [float(i) for i in range(4, 11)]
    hash_max_nota = hash_notas['Nota'].max()
    hash_notas_posibles = hash_notas_2026_en_adelante if pd.isna(hash_max_nota) or hash_max_nota > 2 else hash_notas_pre_2026

    # Cuenta las ocurrencias por nota, completando con 0 las notas posibles que nadie sacó
    hash_notas_count = hash_notas['Nota'].value_counts().reindex(hash_notas_posibles, fill_value=0)

    # Calcula el total y el promedio de los aprobados
    hash_notas_students_count = hash_notas_count.sum()
    hash_notas_mean = hash_notas['Nota'].mean()

##### Resultado

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_notas_bar_chart(hash_notas_count, hash_notas_students_count, hash_notas_mean, fig_num=f"{SECTION_FIGURE_PREFIX['HASH']}.2")

#### Notas por sección de HASH

Este módulo desglosa la nota final de quienes aprobaron HASH en sus tres componentes — `Codigo`, `Pruebas` e `Informe` — para ver si el rendimiento es parejo entre secciones o si alguna concentra las notas más bajas, replicando el mismo análisis de Notas por sección del TP1.

- Parte de la misma población que Notas de HASH (aprobados con nota numérica cargada) y del mismo esquema de notas ya detectado para `Nota final`, ya que las tres secciones puntúan en la misma escala (0 a 2 antes de 2026, 4 a 10 desde 2026).
- Al igual que Notas de HASH, siempre muestra todas las notas posibles del esquema vigente en cada sección, aunque nadie haya sacado esa nota, para que el gráfico no cambie de forma de un cuatrimestre a otro.

Como resultado, el módulo genera una figura con tres gráficos de barras — uno por sección, con el mismo eje Y para poder comparar alturas entre ellos — más el promedio de cada sección en la caja de referencias.

##### Implementación

Esta función arma una figura de tres subplots (uno por sección) y delega el dibujo de cada barra en `_draw_count_bar` (ver Utilidades), el mismo helper que usa Notas de HASH — así evita repetir la lógica de la rampa de color, la grilla y los ejes tantas veces como secciones haya. Solo agrega el título de figura, el eje Y compartido, y una caja de referencias única con el promedio de cada sección.

In [ ]:
def hash_notas_por_seccion_bar_chart(secciones_counts, secciones_means, students_count, fig_num):
  secciones_labels = {'Codigo': 'Código', 'Pruebas': 'Pruebas', 'Informe': 'Informe'}

  fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), sharey=True)

  for ax, col in zip(axes, secciones_counts.keys()):
      _draw_count_bar(ax, secciones_counts[col], students_count, xlabel='Nota', title=secciones_labels[col])

  for ax in axes[1:]:
      ax.set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise a los
  # demas paneles; se recalcula una sola vez con el maximo real de las 3 secciones.
  max_count = max(counts.values.max() for counts in secciones_counts.values())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas por sección de HASH',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas de HASH), con un
  # promedio por sección en vez de uno solo.
  legend_labels = [f"Total: {students_count} aprobados"] + [
      f"Promedio {secciones_labels[col]}: {mean:.1f}" if students_count > 0 else f"Promedio {secciones_labels[col]}: N/A"
      for col, mean in secciones_means.items()
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.06, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.01,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.22, right=0.83, wspace=0.18)
  plt.show()

Esta sección filtra a los mismos aprobados que Notas de HASH, convierte Codigo, Pruebas e Informe a numérico (mismo formato de coma decimal), y cuenta cuántos alumnos sacaron cada nota en cada sección, reutilizando el `hash_notas_posibles` ya calculado para `Nota final`.

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_secciones = sheets[HASH_SHEET][['Padron', 'Nota final'] + ['Codigo', 'Pruebas', 'Informe']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas), mismo filtro que hash_notas
    hash_secciones = hash_secciones[~hash_secciones['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    hash_secciones_cols = ['Codigo', 'Pruebas', 'Informe']
    for col in hash_secciones_cols:
        # Mismo formato de coma decimal que el resto de las notas
        hash_secciones[col] = pd.to_numeric(hash_secciones[col].str.replace(',', '.', regex=False))

    # Reutiliza el esquema de notas ya detectado para "Nota final" (mismas escalas: 0-2 antes
    # de 2026, 4-10 desde 2026), ya que todas las secciones puntúan en la misma escala
    hash_secciones_counts = {
        col: hash_secciones[col].value_counts().reindex(hash_notas_posibles, fill_value=0)
        for col in hash_secciones_cols
    }
    hash_secciones_means = {col: hash_secciones[col].mean() for col in hash_secciones_cols}
    hash_secciones_students_count = len(hash_secciones)

##### Resultado

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_notas_por_seccion_bar_chart(hash_secciones_counts, hash_secciones_means, hash_secciones_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['HASH']}.3")

#### Resultado en el TP anterior de los desaprobados de HASH

Este módulo analiza el desempeño previo (opcional) de quienes desaprobaron HASH en el trabajo práctico anterior, replicando el mismo análisis de TP1 Desaprobados. A diferencia del resto de las secciones, ese "trabajo anterior" no es siempre el mismo: es **ABB** si esa hoja está disponible en esta planilla, o **LISTA** si no lo está (`ABB` dejó de tomarse desde 2026 — ver `hoja_disponible` en Utilidades) — así el módulo salta el hueco en la secuencia en vez de comparar siempre contra una hoja que puede no existir.

- Aísla a la población de estudiantes que desaprobó HASH y, a partir de esos registros, cruza su resultado en ese TP anterior (derivado de su columna `Nota final`, con la misma lógica que Resultados de HASH) para determinar si aprobaron, desaprobaron, todavía no tienen nota, o nunca entregaron ese trabajo.
- Identifica a los estudiantes que están presentes en `HASH` pero ausentes en los registros del TP anterior elegido, y los clasifica como *"No disponible"* (N/A). Esto es distinto de **Sin Nota**: un estudiante que sí está en ese TP pero cuya `Nota final` todavía está vacía o en "Reentrega" — antes ambos casos se mezclaban bajo N/A; separarlos evita confundir "no entregó" con "entregó pero todavía no tiene resultado".
- También asegura que el análisis siempre considere los cuatro resultados posibles de ese TP (**Aprobado, Desaprobado, Sin Nota, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si ni `ABB` ni `LISTA` están disponibles en esta planilla (caso extremo, ya que `LISTA` es una de las `HOJAS_REQUERIDAS`), este módulo igual corre: todos los estudiantes caen directamente en N/A, igual que si cada uno individualmente no tuviera registro en el TP elegido.

Como resultado, el módulo devuelve un DataFrame con los datos de esos estudiantes, más un donut para una mejor visualización de los números, con el total de desaprobados en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo, delegando el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la misma paleta de estado (`STATUS_COLORS`) que Resultados de HASH. A diferencia del resto de los gráficos de esta sección, recibe el nombre del TP anterior elegido (`prev_name`, `'ABB'` o `'LISTA'`) como parámetro para armar el título, en vez de tenerlo hardcodeado, ya que ese nombre cambia según la planilla.

In [ ]:
def failed_hash_donut_chart(categories, categories_count, students_count, prev_name, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning'], STATUS_COLORS['neutral']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      f'Resultado en {prev_name} de los desaprobados de HASH',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que desaprobaron, descarta las columnas de notas (`Codigo`, `Pruebas`, `Informe`, `Nota final`) ya que no aportan nada al estar todos los registros desaprobados, elige el TP anterior (`ABB` si está disponible, si no `LISTA`) y categoriza esos registros según ese TP (misma derivación desde `Nota final` que en Resultados de HASH).

In [ ]:
if hoja_disponible[HASH_SHEET]:
    # El TP anterior a HASH es ABB, salvo que esa hoja no exista en esta planilla (dejó de
    # tomarse desde 2026): en ese caso el anterior pasa a ser LISTA.
    hash_prev_sheet = ABB_SHEET if hoja_disponible[ABB_SHEET] else LISTA_SHEET

    failed_hash_categories = [
        f'{hash_prev_sheet} Aprobado', f'{hash_prev_sheet} Desaprobado',
        f'{hash_prev_sheet} Sin Nota', f'{hash_prev_sheet} N/A'
    ]

    # Filtra a los estudiantes que desaprobaron HASH
    failed_hash = sheets[HASH_SHEET][sheets[HASH_SHEET]['Nota final'] == 'Desaprobado'].copy()

    # Descarta las columnas de notas: no aportan nada ya que todos los registros ya son desaprobados
    failed_hash.drop(columns=['Codigo', 'Pruebas', 'Informe', 'Nota final'], errors='ignore', inplace=True)

    # Deriva el resultado del TP anterior elegido a partir de su "Nota final" (misma lógica que
    # Resultados de HASH). Si esa hoja no está disponible en esta planilla, se usa un DataFrame
    # vacío: el merge no encuentra coincidencias y todos los estudiantes caen en N/A más abajo.
    hash_prev_previo = sheets.get(hash_prev_sheet, pd.DataFrame(columns=['Padron', 'Nota final']))[['Padron', 'Nota final']].copy()
    hash_prev_previo['Prev_Final'] = hash_prev_previo['Nota final'].apply(
        lambda nota: f'{hash_prev_sheet} Desaprobado' if nota == 'Desaprobado' else (f'{hash_prev_sheet} Sin Nota' if nota in ('', 'Reentrega') else f'{hash_prev_sheet} Aprobado')
    )

    failed_hash = failed_hash.merge(
        hash_prev_previo[['Padron', 'Prev_Final']],
        on='Padron',
        how='left'
    )
    failed_hash['Prev_Final'] = failed_hash['Prev_Final'].fillna(f'{hash_prev_sheet} N/A')

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 4 categorías (incluso si el conteo es 0)
    failed_hash_categories_count = failed_hash['Prev_Final'].value_counts().reindex(failed_hash_categories, fill_value=0)

    # Calcula el total de estudiantes
    failed_hash_students_count = failed_hash_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[HASH_SHEET]:
    display(failed_hash)
    print(f"\n")
    failed_hash_donut_chart(failed_hash_categories, failed_hash_categories_count, failed_hash_students_count, hash_prev_sheet, fig_num=f"{SECTION_FIGURE_PREFIX['HASH']}.4")

#### Intentos en HASH

Este módulo compara la cantidad de `intentos` de entrega entre quienes aprobaron y quienes desaprobaron HASH, replicando el mismo análisis de Intentos en TP1, para ver si un mayor número de reintentos se asocia con el resultado final.

- Como `intentos` puede tomar muchos valores distintos con pocas ocurrencias cada uno, se agrupa en rangos fijos (`1-3`, `4-6`, `7-10`, `11-15`, `16-20`, `21+`) en vez de graficar un valor por barra.
- Al igual que en Intentos en TP1, siempre se muestran los seis rangos aunque alguno tenga 0 alumnos, para que el gráfico no cambie de forma de un cuatrimestre a otro.
- El promedio que se muestra en la caja de Referencias se calcula sobre los valores reales de `intentos` (sin binnear), no sobre los rangos.

Como resultado, el módulo genera una figura con dos gráficos de barras — Aprobados y Desaprobados, con el mismo rango de bins en el eje X — más el total y el promedio de cada grupo en la caja de referencias.

##### Implementación

También delega el dibujo de cada barra en `_draw_count_bar`, esta vez sobre una figura de dos subplots (Aprobados y Desaprobados) que comparten el eje Y y el mismo rango de bins en el eje X, para que las alturas sean directamente comparables entre los dos grupos.

In [ ]:
def hash_intentos_bar_chart(counts_aprobados, total_aprobados, mean_aprobados, counts_desaprobados, total_desaprobados, mean_desaprobados, fig_num):
  fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

  _draw_count_bar(axes[0], counts_aprobados, total_aprobados, xlabel='Intentos', title='Aprobados')
  _draw_count_bar(axes[1], counts_desaprobados, total_desaprobados, xlabel='Intentos', title='Desaprobados')
  axes[1].set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise al otro
  # panel; se recalcula una sola vez con el maximo real de ambos grupos.
  max_count = max(counts_aprobados.values.max(), counts_desaprobados.values.max())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Intentos en HASH',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  promedio_aprobados_txt = f"{mean_aprobados:.1f}" if total_aprobados > 0 else "N/A"
  promedio_desaprobados_txt = f"{mean_desaprobados:.1f}" if total_desaprobados > 0 else "N/A"
  legend_labels = [
      f"Aprobados: {total_aprobados} (prom. {promedio_aprobados_txt})",
      f"Desaprobados: {total_desaprobados} (prom. {promedio_desaprobados_txt})",
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.24, right=0.68, wspace=0.12)
  plt.show()

Esta sección separa HASH en aprobados (mismo filtro que Notas de HASH) y desaprobados (mismo filtro que HASH Desaprobados), convierte `intentos` a numérico, y cuenta cuántos alumnos de cada grupo caen en cada rango de intentos.

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_intentos_bins = [0, 3, 6, 10, 15, 20, float('inf')]
    hash_intentos_labels = ['1-3', '4-6', '7-10', '11-15', '16-20', '21+']

    # Aprobados: mismo filtro que hash_notas
    hash_intentos_aprobados = sheets[HASH_SHEET][~sheets[HASH_SHEET]['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])].copy()
    hash_intentos_aprobados['intentos'] = pd.to_numeric(hash_intentos_aprobados['intentos'].str.replace(',', '.', regex=False))
    hash_intentos_aprobados_count = pd.cut(
        hash_intentos_aprobados['intentos'], bins=hash_intentos_bins, labels=hash_intentos_labels
    ).value_counts().reindex(hash_intentos_labels, fill_value=0)
    hash_intentos_aprobados_total = len(hash_intentos_aprobados)
    hash_intentos_aprobados_mean = hash_intentos_aprobados['intentos'].mean()

    # Desaprobados: mismo filtro que failed_hash
    hash_intentos_desaprobados = sheets[HASH_SHEET][sheets[HASH_SHEET]['Nota final'] == 'Desaprobado'].copy()
    hash_intentos_desaprobados['intentos'] = pd.to_numeric(hash_intentos_desaprobados['intentos'].str.replace(',', '.', regex=False))
    hash_intentos_desaprobados_count = pd.cut(
        hash_intentos_desaprobados['intentos'], bins=hash_intentos_bins, labels=hash_intentos_labels
    ).value_counts().reindex(hash_intentos_labels, fill_value=0)
    hash_intentos_desaprobados_total = len(hash_intentos_desaprobados)
    hash_intentos_desaprobados_mean = hash_intentos_desaprobados['intentos'].mean()

##### Resultado

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_intentos_bar_chart(
        hash_intentos_aprobados_count, hash_intentos_aprobados_total, hash_intentos_aprobados_mean,
        hash_intentos_desaprobados_count, hash_intentos_desaprobados_total, hash_intentos_desaprobados_mean,
        fig_num=f"{SECTION_FIGURE_PREFIX['HASH']}.5"
    )

#### Estado en HASH Desaprobados

Este módulo mira el `Estado` que el pipeline de corrección automática le asignó a cada desaprobado de HASH, replicando el mismo análisis de Estado en TP1 Desaprobados, para distinguir cuántos fallaron por un problema del propio pipeline (no compila, timeout) de cuántos sí llegaron a corregirse pero no alcanzaron la nota mínima.

- Normaliza los valores de `Estado`, que cambian de vocabulario entre esquemas de planilla (`ok`/`error` en 2025, `SUCCESS`/`FAILED`/`TIMEOUT` desde 2026), a tres categorías fijas: **OK**, **Timeout** y **Error**.
- Siempre muestra las tres categorías con su propia barra aunque alguna tenga 0 alumnos, por la misma razón que en Estado en TP1 Desaprobados: un gráfico de torta esconde o directamente hace desaparecer las categorías en 0.
- A diferencia de los demás gráficos de barras de la notebook, que colorean según la magnitud de cada barra, acá el color es fijo por categoría — la paleta de estado (verde/ámbar/rojo) — porque comunica severidad, no una magnitud relativa.

Como resultado, el módulo genera un gráfico de barras con la proporción de cada estado entre los desaprobados.

##### Implementación

Esta función reutiliza `_draw_count_bar` (ver Utilidades) igual que Notas de HASH, pero le pasa una lista de colores fija en vez de dejar que calcule la rampa secuencial — la paleta de estado (`good`/`warning`/`critical`, verde/ámbar/rojo), ya que acá el color identifica cada estado y no una magnitud.

In [ ]:
def hash_estado_bar_chart(categories_count, students_count, fig_num):
  plt.figure(figsize=(9, 5.5))
  ax = plt.gca()

  # Paleta de estado (good/warning/critical) en vez de la rampa secuencial de
  # _draw_count_bar: acá el color identifica cada estado, no una magnitud.
  status_colors = {'OK': '#0ca30c', 'Timeout': '#fab219', 'Error': '#d03b3b'}
  bar_colors = [status_colors[cat] for cat in categories_count.index]

  _draw_count_bar(ax, categories_count, students_count, xlabel='Estado', colors=bar_colors)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Estado en HASH Desaprobados',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que Notas de HASH: las barras ya muestran cantidad y
  # porcentaje de cada estado, así que la caja solo necesita el total.
  legend_labels = [f"Total: {students_count} desaprobados"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.82, bottom=0.24, right=0.62)
  plt.show()

Esta sección filtra a los desaprobados de HASH (mismo filtro que HASH Desaprobados) y mapea su `Estado` a las tres categorías normalizadas, usando `_normalize_key` para que el mapeo no dependa de mayúsculas/minúsculas.

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_estado_categories = ['OK', 'Timeout', 'Error']

    # El pipeline de corrección devuelve vocabularios distintos según el esquema de planilla:
    # 'ok'/'error' en 2025, 'SUCCESS'/'FAILED'/'TIMEOUT' desde 2026. Se normalizan a 3 categorías fijas.
    hash_status_aliases = {'ok': 'OK', 'success': 'OK', 'error': 'Error', 'failed': 'Error', 'timeout': 'Timeout'}

    hash_estado = sheets[HASH_SHEET][sheets[HASH_SHEET]['Nota final'] == 'Desaprobado'][['Padron', 'Estado']].copy()
    hash_estado_crudo = hash_estado['Estado']
    hash_estado['Estado'] = hash_estado_crudo.map(lambda s: hash_status_aliases.get(_normalize_key(s)))
    _warn_unmapped(hash_estado['Estado'], hash_estado_crudo, "Estado en HASH Desaprobados: columna 'Estado'")

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    hash_estado_categories_count = hash_estado['Estado'].value_counts().reindex(hash_estado_categories, fill_value=0)

    # Calcula el total de estudiantes
    hash_estado_students_count = hash_estado_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[HASH_SHEET]:
    hash_estado_bar_chart(hash_estado_categories_count, hash_estado_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['HASH']}.6")

### TP2

Estadísticas basadas en la hoja `TP2`, el sexto y último trabajo práctico de la materia. Al igual que en HASH, el módulo Desaprobados cruza el resultado de TP2 con el de HASH, tratándolo como un antecedente opcional. A diferencia de LISTA, ABB y HASH, la hoja `TP2` suma una cuarta sección de nota — `Interaccion`, la defensa oral — además de `Codigo`, `Pruebas` e `Informe`.

#### Resultados del TP2

Este módulo calcula la proporción de estudiantes que aprobaron, desaprobaron, o todavía no tienen nota en el TP2, replicando el mismo análisis de Resultados del TP1 (y RESULTADOS TP0) para el sexto y último trabajo práctico de la materia.

- Clasifica a cada estudiante presente en la hoja `TP2` según su columna `Nota final`, en **Aprobado** (si tiene una nota numérica cargada), **Desaprobado** (si el valor es el string "Desaprobado"), o **Sin Nota** (si `Nota final` está vacío o marcado como "Reentrega" — corrección todavía abierta, o pendiente de una nueva entrega).

Como resultado, el módulo genera un donut para visualizar la proporción de cada categoría, con el total de alumnos en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo. Delega el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la paleta de estado fija (`STATUS_COLORS`), el mismo esquema que usan el resto de los módulos de esta sección y de TP1. Arma el título y la caja de Referencias; el propio donut ya muestra el total en su hueco central.

In [ ]:
def tp2_results_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados del TP2',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes del TP2 según su resultado en `Nota final`, con la misma lógica que Resultados del TP1.

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_results_categories = ['Aprobado', 'Desaprobado', 'Sin Nota']

    tp2_results = sheets[TP2_SHEET][['Padron', 'Nota final']].copy()

    # Clasifica según "Nota final": el string "Desaprobado", vacío o "Reentrega" (corrección todavía abierta o pendiente de una nueva entrega), o una nota numérica
    tp2_results['Resultado'] = tp2_results['Nota final'].apply(
        lambda nota: 'Desaprobado' if nota == 'Desaprobado' else ('Sin Nota' if nota in ('', 'Reentrega') else 'Aprobado')
    )
    tp2_results.drop(columns=['Nota final'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    tp2_results_categories_count = tp2_results['Resultado'].value_counts().reindex(tp2_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp2_results_students_count = tp2_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_results_donut_chart(tp2_results_categories, tp2_results_categories_count, tp2_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP2']}.1")

#### Notas del TP2

Este módulo muestra la distribución de las notas numéricas obtenidas por quienes aprobaron el TP2, replicando el mismo análisis de Notas del TP1.

- Toma únicamente los registros de `TP2` con una nota numérica cargada, descartando los desaprobados (`Nota final == "Desaprobado"`) y las correcciones todavía abiertas o en reentrega (`Nota final` vacío o "Reentrega").
- Siempre muestra **todas** las notas posibles del esquema vigente, aunque nadie haya sacado esa nota (con 0 alumnos), para que el gráfico no cambie de forma según qué notas salieron ese cuatrimestre.
- Cada barra se colorea con un degradé secuencial según su propia cantidad de alumnos (más clara = menos alumnos, más oscura = más alumnos), para que la nota más frecuente salte a la vista.

Como resultado, el módulo genera un gráfico de barras con el porcentaje y la cantidad de alumnos que sacó cada nota, más el total y el promedio en la caja de referencias.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de barras de este módulo. La rampa de color ordinal y el resto de la estética de cada barra viven en `_draw_count_bar` (ver Utilidades), igual que en Notas del TP1; esta función arma la figura, el título, y la caja de referencias con el total y el promedio.

In [ ]:
def tp2_notas_bar_chart(notas_count, students_count, mean_nota, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, notas_count, students_count, xlabel='Nota')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas del TP2',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas del TP1): sin categorías con
  # color que mostrar, los handles son invisibles (handlelength=0) y solo quedan las dos líneas de texto.
  promedio_txt = f"{mean_nota:.1f}" if students_count > 0 else "N/A"
  legend_labels = [f"Total: {students_count} aprobados", f"Promedio: {promedio_txt}"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.6)
  plt.show()

Esta sección filtra a los estudiantes que aprobaron el TP2, cuenta cuántos sacaron cada nota completando con 0 las notas del esquema vigente que nadie sacó, y calcula el promedio, reutilizando la misma detección de esquema (0-2 antes de 2026, 4-10 desde 2026) que Notas del TP1.

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_notas = sheets[TP2_SHEET][['Padron', 'Nota final']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas)
    tp2_notas = tp2_notas[~tp2_notas['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    # Google Sheets devuelve los decimales con coma (ej. "0,75") en planillas con configuración regional en español
    tp2_notas['Nota'] = pd.to_numeric(tp2_notas['Nota final'].str.replace(',', '.', regex=False))

    # Antes de 2026 las notas iban de 0 a 2 en intervalos de 0.25; desde 2026 van de 4 a 10 en intervalos de 1.
    # Como los rangos no se solapan, la nota máxima observada alcanza para inferir qué esquema aplica.
    tp2_notas_pre_2026 = [round(i * 0.25, 2) for i in range(9)]
    tp2_notas_2026_en_adelante = [float(i) for i in range(4, 11)]
    tp2_max_nota = tp2_notas['Nota'].max()
    tp2_notas_posibles = tp2_notas_2026_en_adelante if pd.isna(tp2_max_nota) or tp2_max_nota > 2 else tp2_notas_pre_2026

    # Cuenta las ocurrencias por nota, completando con 0 las notas posibles que nadie sacó
    tp2_notas_count = tp2_notas['Nota'].value_counts().reindex(tp2_notas_posibles, fill_value=0)

    # Calcula el total y el promedio de los aprobados
    tp2_notas_students_count = tp2_notas_count.sum()
    tp2_notas_mean = tp2_notas['Nota'].mean()

##### Resultado

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_notas_bar_chart(tp2_notas_count, tp2_notas_students_count, tp2_notas_mean, fig_num=f"{SECTION_FIGURE_PREFIX['TP2']}.2")

#### Notas por sección del TP2

Este módulo desglosa la nota final de quienes aprobaron el TP2 en sus cuatro componentes — `Codigo`, `Pruebas`, `Informe` e `Interaccion` — para ver si el rendimiento es parejo entre secciones o si alguna concentra las notas más bajas. A diferencia de Notas por sección del TP1, acá se suma `Interaccion` (la defensa oral), la única sección propia del TP2.

- Parte de la misma población que Notas del TP2 (aprobados con nota numérica cargada) y del mismo esquema de notas ya detectado para `Nota final`, ya que las cuatro secciones puntúan en la misma escala (0 a 2 antes de 2026, 4 a 10 desde 2026).
- Al igual que Notas del TP2, siempre muestra todas las notas posibles del esquema vigente en cada sección, aunque nadie haya sacado esa nota, para que el gráfico no cambie de forma de un cuatrimestre a otro.

Como resultado, el módulo genera una figura con cuatro gráficos de barras — uno por sección, con el mismo eje Y para poder comparar alturas entre ellos — más el promedio de cada sección en la caja de referencias.

##### Implementación

Esta función arma una figura de cuatro subplots (uno por sección) y delega el dibujo de cada barra en `_draw_count_bar` (ver Utilidades), el mismo helper que usa Notas del TP2 — así evita repetir la lógica de la rampa de color, la grilla y los ejes tantas veces como secciones haya. Solo agrega el título de figura, el eje Y compartido, y una caja de referencias única con el promedio de cada sección.

In [ ]:
def tp2_notas_por_seccion_bar_chart(secciones_counts, secciones_means, students_count, fig_num):
  secciones_labels = {'Codigo': 'Código', 'Pruebas': 'Pruebas', 'Informe': 'Informe', 'Interaccion': 'Interacción'}

  fig, axes = plt.subplots(1, 4, figsize=(28, 6.5), sharey=True)

  for ax, col in zip(axes, secciones_counts.keys()):
      _draw_count_bar(ax, secciones_counts[col], students_count, xlabel='Nota', title=secciones_labels[col])

  for ax in axes[1:]:
      ax.set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise a los
  # demas paneles; se recalcula una sola vez con el maximo real de las 4 secciones.
  max_count = max(counts.values.max() for counts in secciones_counts.values())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas por sección del TP2',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas del TP2), con un
  # promedio por sección en vez de uno solo.
  legend_labels = [f"Total: {students_count} aprobados"] + [
      f"Promedio {secciones_labels[col]}: {mean:.1f}" if students_count > 0 else f"Promedio {secciones_labels[col]}: N/A"
      for col, mean in secciones_means.items()
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.06, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.01,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.22, right=0.83, wspace=0.18)
  plt.show()

Esta sección filtra a los mismos aprobados que Notas del TP2, convierte Codigo, Pruebas, Informe e Interaccion a numérico (mismo formato de coma decimal), y cuenta cuántos alumnos sacaron cada nota en cada sección, reutilizando el `tp2_notas_posibles` ya calculado para `Nota final`.

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_secciones = sheets[TP2_SHEET][['Padron', 'Nota final'] + ['Codigo', 'Pruebas', 'Informe', 'Interaccion']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado", "Reentrega" y correcciones abiertas), mismo filtro que tp2_notas
    tp2_secciones = tp2_secciones[~tp2_secciones['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])]

    tp2_secciones_cols = ['Codigo', 'Pruebas', 'Informe', 'Interaccion']
    for col in tp2_secciones_cols:
        # Mismo formato de coma decimal que el resto de las notas
        tp2_secciones[col] = pd.to_numeric(tp2_secciones[col].str.replace(',', '.', regex=False))

    # Reutiliza el esquema de notas ya detectado para "Nota final" (mismas escalas: 0-2 antes
    # de 2026, 4-10 desde 2026), ya que todas las secciones puntúan en la misma escala
    tp2_secciones_counts = {
        col: tp2_secciones[col].value_counts().reindex(tp2_notas_posibles, fill_value=0)
        for col in tp2_secciones_cols
    }
    tp2_secciones_means = {col: tp2_secciones[col].mean() for col in tp2_secciones_cols}
    tp2_secciones_students_count = len(tp2_secciones)

##### Resultado

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_notas_por_seccion_bar_chart(tp2_secciones_counts, tp2_secciones_means, tp2_secciones_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP2']}.3")

#### Resultado en HASH de los desaprobados de TP2

Este módulo analiza el desempeño previo (opcional) en HASH de quienes desaprobaron el TP2, replicando el mismo análisis de TP1 Desaprobados. Esto puede darnos una idea de si desaprobar el TP2 se relaciona con haber tenido dificultades ya en el trabajo anterior.

- Aísla a la población de estudiantes que desaprobó el TP2 y, a partir de esos registros, cruza su resultado en HASH (derivado de su columna `Nota final`, con la misma lógica que Resultados del TP2) para determinar si aprobaron, desaprobaron, todavía no tienen nota, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en `TP2` pero ausentes en los registros de `HASH`, y los clasifica como *"No disponible"* (N/A). Esto es distinto de **Sin Nota**: un estudiante que sí está en `HASH` pero cuya `Nota final` todavía está vacía o en "Reentrega" — antes ambos casos se mezclaban bajo N/A; separarlos evita confundir "no entregó HASH" con "entregó HASH pero todavía no tiene resultado".
- También asegura que el análisis siempre considere los cuatro resultados posibles de `HASH` (**Aprobado, Desaprobado, Sin Nota, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si la hoja `HASH` no está disponible en esta planilla (a diferencia del resto de las secciones, que se omiten enteras cuando falta la hoja que necesitan — ver `hoja_disponible` en Utilidades), este módulo igual corre: todos los estudiantes caen directamente en **HASH N/A**, igual que si cada uno individualmente no tuviera registro en `HASH`.

Como resultado, el módulo devuelve un DataFrame con los datos de esos estudiantes, más un donut para una mejor visualización de los números, con el total de desaprobados en el centro.

##### Implementación

Esta función gestiona la generación y visualización completa del donut de este módulo, delegando el dibujo de las porciones en `_draw_status_donut` (ver Utilidades) con la misma paleta de estado (`STATUS_COLORS`) que Resultados del TP2.

In [ ]:
def failed_tp2_donut_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))
  ax = plt.gca()

  colors = [STATUS_COLORS['good'], STATUS_COLORS['critical'], STATUS_COLORS['warning'], STATUS_COLORS['neutral']]
  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  patches = _draw_status_donut(ax, categories, categories_count, students_count, colors)

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  ax.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultado en HASH de los desaprobados de TP2',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)
  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que desaprobaron, descarta las columnas de notas (`Codigo`, `Pruebas`, `Informe`, `Interaccion`, `Nota final`) ya que no aportan nada al estar todos los registros desaprobados, y categoriza esos registros según su HASH (misma derivación desde `Nota final` que en Resultados del TP2).

In [ ]:
if hoja_disponible[TP2_SHEET]:
    failed_tp2_categories = ['HASH Aprobado', 'HASH Desaprobado', 'HASH Sin Nota', 'HASH N/A']

    # Filtra a los estudiantes que desaprobaron el TP2
    failed_tp2 = sheets[TP2_SHEET][sheets[TP2_SHEET]['Nota final'] == 'Desaprobado'].copy()

    # Descarta las columnas de notas: no aportan nada ya que todos los registros ya son desaprobados
    failed_tp2.drop(columns=['Codigo', 'Pruebas', 'Informe', 'Interaccion', 'Nota final'], errors='ignore', inplace=True)

    # Deriva el resultado de HASH a partir de su "Nota final" (misma lógica que Resultados del TP2).
    # Si la hoja HASH no está disponible en esta planilla, se usa un DataFrame vacío: el merge no
    # encuentra coincidencias y todos los estudiantes caen en "HASH N/A" más abajo.
    tp2_hash_previo = sheets.get(HASH_SHEET, pd.DataFrame(columns=['Padron', 'Nota final']))[['Padron', 'Nota final']].copy()
    tp2_hash_previo['HASH_Final'] = tp2_hash_previo['Nota final'].apply(
        lambda nota: 'HASH Desaprobado' if nota == 'Desaprobado' else ('HASH Sin Nota' if nota in ('', 'Reentrega') else 'HASH Aprobado')
    )

    failed_tp2 = failed_tp2.merge(
        tp2_hash_previo[['Padron', 'HASH_Final']],
        on='Padron',
        how='left'
    )
    failed_tp2['HASH_Final'] = failed_tp2['HASH_Final'].fillna('HASH N/A')

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 4 categorías (incluso si el conteo es 0)
    failed_tp2_categories_count = failed_tp2['HASH_Final'].value_counts().reindex(failed_tp2_categories, fill_value=0)

    # Calcula el total de estudiantes
    failed_tp2_students_count = failed_tp2_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP2_SHEET]:
    display(failed_tp2)
    print(f"\n")
    failed_tp2_donut_chart(failed_tp2_categories, failed_tp2_categories_count, failed_tp2_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP2']}.4")

#### Intentos en TP2

Este módulo compara la cantidad de `intentos` de entrega entre quienes aprobaron y quienes desaprobaron el TP2, replicando el mismo análisis de Intentos en TP1, para ver si un mayor número de reintentos se asocia con el resultado final.

- Como `intentos` puede tomar muchos valores distintos con pocas ocurrencias cada uno, se agrupa en rangos fijos (`1-3`, `4-6`, `7-10`, `11-15`, `16-20`, `21+`) en vez de graficar un valor por barra.
- Al igual que en Intentos en TP1, siempre se muestran los seis rangos aunque alguno tenga 0 alumnos, para que el gráfico no cambie de forma de un cuatrimestre a otro.
- El promedio que se muestra en la caja de Referencias se calcula sobre los valores reales de `intentos` (sin binnear), no sobre los rangos.

Como resultado, el módulo genera una figura con dos gráficos de barras — Aprobados y Desaprobados, con el mismo rango de bins en el eje X — más el total y el promedio de cada grupo en la caja de referencias.

##### Implementación

También delega el dibujo de cada barra en `_draw_count_bar`, esta vez sobre una figura de dos subplots (Aprobados y Desaprobados) que comparten el eje Y y el mismo rango de bins en el eje X, para que las alturas sean directamente comparables entre los dos grupos.

In [ ]:
def tp2_intentos_bar_chart(counts_aprobados, total_aprobados, mean_aprobados, counts_desaprobados, total_desaprobados, mean_desaprobados, fig_num):
  fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

  _draw_count_bar(axes[0], counts_aprobados, total_aprobados, xlabel='Intentos', title='Aprobados')
  _draw_count_bar(axes[1], counts_desaprobados, total_desaprobados, xlabel='Intentos', title='Desaprobados')
  axes[1].set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise al otro
  # panel; se recalcula una sola vez con el maximo real de ambos grupos.
  max_count = max(counts_aprobados.values.max(), counts_desaprobados.values.max())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Intentos en TP2',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  promedio_aprobados_txt = f"{mean_aprobados:.1f}" if total_aprobados > 0 else "N/A"
  promedio_desaprobados_txt = f"{mean_desaprobados:.1f}" if total_desaprobados > 0 else "N/A"
  legend_labels = [
      f"Aprobados: {total_aprobados} (prom. {promedio_aprobados_txt})",
      f"Desaprobados: {total_desaprobados} (prom. {promedio_desaprobados_txt})",
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.24, right=0.68, wspace=0.12)
  plt.show()

Esta sección separa el TP2 en aprobados (mismo filtro que Notas del TP2) y desaprobados (mismo filtro que TP2 Desaprobados), convierte `intentos` a numérico, y cuenta cuántos alumnos de cada grupo caen en cada rango de intentos.

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_intentos_bins = [0, 3, 6, 10, 15, 20, float('inf')]
    tp2_intentos_labels = ['1-3', '4-6', '7-10', '11-15', '16-20', '21+']

    # Aprobados: mismo filtro que tp2_notas
    tp2_intentos_aprobados = sheets[TP2_SHEET][~sheets[TP2_SHEET]['Nota final'].isin(['', 'Desaprobado', 'Reentrega'])].copy()
    tp2_intentos_aprobados['intentos'] = pd.to_numeric(tp2_intentos_aprobados['intentos'].str.replace(',', '.', regex=False))
    tp2_intentos_aprobados_count = pd.cut(
        tp2_intentos_aprobados['intentos'], bins=tp2_intentos_bins, labels=tp2_intentos_labels
    ).value_counts().reindex(tp2_intentos_labels, fill_value=0)
    tp2_intentos_aprobados_total = len(tp2_intentos_aprobados)
    tp2_intentos_aprobados_mean = tp2_intentos_aprobados['intentos'].mean()

    # Desaprobados: mismo filtro que failed_tp2
    tp2_intentos_desaprobados = sheets[TP2_SHEET][sheets[TP2_SHEET]['Nota final'] == 'Desaprobado'].copy()
    tp2_intentos_desaprobados['intentos'] = pd.to_numeric(tp2_intentos_desaprobados['intentos'].str.replace(',', '.', regex=False))
    tp2_intentos_desaprobados_count = pd.cut(
        tp2_intentos_desaprobados['intentos'], bins=tp2_intentos_bins, labels=tp2_intentos_labels
    ).value_counts().reindex(tp2_intentos_labels, fill_value=0)
    tp2_intentos_desaprobados_total = len(tp2_intentos_desaprobados)
    tp2_intentos_desaprobados_mean = tp2_intentos_desaprobados['intentos'].mean()

##### Resultado

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_intentos_bar_chart(
        tp2_intentos_aprobados_count, tp2_intentos_aprobados_total, tp2_intentos_aprobados_mean,
        tp2_intentos_desaprobados_count, tp2_intentos_desaprobados_total, tp2_intentos_desaprobados_mean,
        fig_num=f"{SECTION_FIGURE_PREFIX['TP2']}.5"
    )

#### Estado en TP2 Desaprobados

Este módulo mira el `Estado` que el pipeline de corrección automática le asignó a cada desaprobado del TP2, replicando el mismo análisis de Estado en TP1 Desaprobados, para distinguir cuántos fallaron por un problema del propio pipeline (no compila, timeout) de cuántos sí llegaron a corregirse pero no alcanzaron la nota mínima.

- Normaliza los valores de `Estado`, que cambian de vocabulario entre esquemas de planilla (`ok`/`error` en 2025, `SUCCESS`/`FAILED`/`TIMEOUT` desde 2026), a tres categorías fijas: **OK**, **Timeout** y **Error**.
- Siempre muestra las tres categorías con su propia barra aunque alguna tenga 0 alumnos, por la misma razón que en Estado en TP1 Desaprobados: un gráfico de torta esconde o directamente hace desaparecer las categorías en 0.
- A diferencia de los demás gráficos de barras de la notebook, que colorean según la magnitud de cada barra, acá el color es fijo por categoría — la paleta de estado (verde/ámbar/rojo) — porque comunica severidad, no una magnitud relativa.

Como resultado, el módulo genera un gráfico de barras con la proporción de cada estado entre los desaprobados.

##### Implementación

Esta función reutiliza `_draw_count_bar` (ver Utilidades) igual que Notas del TP2, pero le pasa una lista de colores fija en vez de dejar que calcule la rampa secuencial — la paleta de estado (`good`/`warning`/`critical`, verde/ámbar/rojo), ya que acá el color identifica cada estado y no una magnitud.

In [ ]:
def tp2_estado_bar_chart(categories_count, students_count, fig_num):
  plt.figure(figsize=(9, 5.5))
  ax = plt.gca()

  # Paleta de estado (good/warning/critical) en vez de la rampa secuencial de
  # _draw_count_bar: acá el color identifica cada estado, no una magnitud.
  status_colors = {'OK': '#0ca30c', 'Timeout': '#fab219', 'Error': '#d03b3b'}
  bar_colors = [status_colors[cat] for cat in categories_count.index]

  _draw_count_bar(ax, categories_count, students_count, xlabel='Estado', colors=bar_colors)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Estado en TP2 Desaprobados',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que Notas del TP2: las barras ya muestran cantidad y
  # porcentaje de cada estado, así que la caja solo necesita el total.
  legend_labels = [f"Total: {students_count} desaprobados"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.82, bottom=0.24, right=0.62)
  plt.show()

Esta sección filtra a los desaprobados del TP2 (mismo filtro que TP2 Desaprobados) y mapea su `Estado` a las tres categorías normalizadas, usando `_normalize_key` para que el mapeo no dependa de mayúsculas/minúsculas.

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_estado_categories = ['OK', 'Timeout', 'Error']

    # El pipeline de corrección devuelve vocabularios distintos según el esquema de planilla:
    # 'ok'/'error' en 2025, 'SUCCESS'/'FAILED'/'TIMEOUT' desde 2026. Se normalizan a 3 categorías fijas.
    tp2_status_aliases = {'ok': 'OK', 'success': 'OK', 'error': 'Error', 'failed': 'Error', 'timeout': 'Timeout'}

    tp2_estado = sheets[TP2_SHEET][sheets[TP2_SHEET]['Nota final'] == 'Desaprobado'][['Padron', 'Estado']].copy()
    tp2_estado_crudo = tp2_estado['Estado']
    tp2_estado['Estado'] = tp2_estado_crudo.map(lambda s: tp2_status_aliases.get(_normalize_key(s)))
    _warn_unmapped(tp2_estado['Estado'], tp2_estado_crudo, "Estado en TP2 Desaprobados: columna 'Estado'")

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    tp2_estado_categories_count = tp2_estado['Estado'].value_counts().reindex(tp2_estado_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp2_estado_students_count = tp2_estado_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP2_SHEET]:
    tp2_estado_bar_chart(tp2_estado_categories_count, tp2_estado_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP2']}.6")